In [ ]:

# ================================================================
# EXPERIMENT 2 — POLYSEMANTICITY DISCOVERY
# Qwen2.5-3B Base | 36 Top-K SAEs
#
# One-pass natural-language feature discovery:
#   1. Load WikiText token cache.
#   2. Run frozen Qwen2.5-3B.
#   3. Encode each layer with its existing SAE.
#   4. Select the 64 most frequently active features per layer.
#   5. Retrieve top 12 activating contexts per selected feature.
#   6. Embed contexts and cluster them semantically.
#   7. Rank candidate polysemantic features.
#
# Outputs:
#   polysemanticity_candidates.csv
#   feature_context_examples.csv
#   experiment_summary.json
# ================================================================

!pip -q install transformers sentence-transformers scikit-learn

import os
import json
import torch
import numpy as np
import pandas as pd

from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm

# ------------------------- CONFIG --------------------------------

ROOT = "/content/drive/MyDrive/Mapping_Feature_Drift"

TOKEN_PATH = os.path.join(ROOT, "data", "wikitext103_tokens.pt")

SAE_DIR = os.path.join(
    ROOT, "sae_research_100k_v2", "saes"
)

OUT = os.path.join(
    ROOT, "sae_research_100k_v2", "experiment_02_polysemanticity"
)
os.makedirs(OUT, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-3B"
REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

SEQ_LEN = 256
BATCH_SIZE = 1
TOP_FEATURES = 64
TOP_CONTEXTS = 12
TOP_K = 32
MIN_CLUSTER_SIZE = 3
MAX_SEQUENCES = 190

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

print("Device:", DEVICE)

# ------------------------- LOAD TOKENS ---------------------------


# ------------------------- LOAD TOKENS ---------------------------

print("\nLoading WikiText-103 tokens...")

from google.colab import drive
from datasets import load_dataset

# Ensure Drive is mounted.
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

os.makedirs(os.path.dirname(TOKEN_PATH), exist_ok=True)

if os.path.exists(TOKEN_PATH):

    print("Using existing token cache:", TOKEN_PATH)

    token_data = torch.load(
        TOKEN_PATH,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(token_data, dict):
        if "input_ids" in token_data:
            token_data = token_data["input_ids"]
        elif "tokens" in token_data:
            token_data = token_data["tokens"]

    tokens = torch.as_tensor(token_data).flatten().long()

else:

    print("Token cache missing. Downloading WikiText-103...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=REVISION
    )

    dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="train"
    )

    texts = [
        text.strip()
        for text in dataset["text"]
        if text.strip()
    ]

    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )

    tokens = torch.tensor(
        [
            token
            for sequence in encoded["input_ids"]
            for token in sequence
        ],
        dtype=torch.long
    )

    # Save cache for future experiments.
    torch.save(tokens, TOKEN_PATH)

    print("New token cache saved:", TOKEN_PATH)

# Prepare complete sequences.
usable = (len(tokens) // SEQ_LEN) * SEQ_LEN

tokens = tokens[:usable].reshape(-1, SEQ_LEN)
tokens = tokens[:MAX_SEQUENCES]

print("Sequences:", len(tokens))
print("Tokens:", tokens.numel())
print("Token cache ready.")


# ------------------------- LOAD MODEL ----------------------------

print("\nLoading Qwen2.5-3B Base...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

# ------------------------- SAE LOADER ----------------------------

def load_sae_weights(layer):
    path = os.path.join(
        SAE_DIR, f"sae_layer_{layer:02d}.pt"
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = checkpoint.get("state_dict", checkpoint)

    # Find tensors by their expected dimensions.
    enc = None
    dec = None
    enc_bias = None
    dec_bias = None
    mean = None

    for key, value in state.items():
        if not torch.is_tensor(value):
            continue

        shape = tuple(value.shape)
        name = key.lower()

        if shape == (2048, 8192) and (
            "enc" in name or enc is None
        ):
            enc = value.float()

        elif shape == (8192, 2048) and (
            "dec" in name or dec is None
        ):
            dec = value.float()

        elif shape == (8192,) and "bias" in name:
            if "enc" in name:
                enc_bias = value.float()
            elif "dec" in name:
                dec_bias = value.float()

        elif shape == (2048,) and (
            "mean" in name or "center" in name
        ):
            mean = value.float()

    if enc is None or dec is None or enc_bias is None:
        raise ValueError(
            f"Could not identify SAE parameters in {path}. "
            f"Checkpoint keys: {list(state.keys())}"
        )

    if dec_bias is None:
        dec_bias = torch.zeros(2048)

    if mean is None:
        mean = torch.zeros(2048)

    return {
        "enc": enc.to(DEVICE),
        "dec": dec.to(DEVICE),
        "enc_bias": enc_bias.to(DEVICE),
        "dec_bias": dec_bias.to(DEVICE),
        "mean": mean.to(DEVICE)
    }


def encode_sae(x, sae):
    # x: [tokens, 2048]
    x = x.float() - sae["mean"]

    hidden = x @ sae["enc"] + sae["enc_bias"]
    hidden = torch.relu(hidden)

    values, indices = torch.topk(
        hidden, k=TOP_K, dim=-1
    )

    sparse = torch.zeros_like(hidden)
    sparse.scatter_(1, indices, values)

    return sparse


# ------------------------- CONTEXT STORAGE -----------------------

# Store sequence-level peak activation and the corresponding
# token position for every feature. Only compact arrays are retained.

feature_counts = {
    layer: np.zeros(8192, dtype=np.int64)
    for layer in range(1, 37)
}

feature_best_scores = {
    layer: np.full((8192, TOP_CONTEXTS), -np.inf,
                   dtype=np.float32)
    for layer in range(1, 37)
}

feature_best_seq = {
    layer: np.full((8192, TOP_CONTEXTS), -1,
                   dtype=np.int32)
    for layer in range(1, 37)
}

feature_best_pos = {
    layer: np.full((8192, TOP_CONTEXTS), -1,
                   dtype=np.int32)
    for layer in range(1, 37)
}

sequence_texts = []

# ------------------------- MAIN EXTRACTION ------------------------

print("\nExtracting natural-language feature contexts...")

with torch.inference_mode():

    for seq_idx in tqdm(range(len(tokens))):

        ids = tokens[seq_idx].unsqueeze(0).to(
            model.get_input_embeddings().weight.device
        )

        # All transformer layer outputs.
        outputs = model(
            input_ids=ids,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.hidden_states

        sequence_text = tokenizer.decode(
            ids[0].cpu(),
            skip_special_tokens=True
        )

        sequence_texts.append(sequence_text)

        for layer in range(1, 37):

            sae = load_sae_weights(layer)

            # hidden_states[0] is embedding output.
            residual = hidden_states[layer][0]

            # Move activation to SAE device.
            residual = residual.to(DEVICE)

            acts = encode_sae(residual, sae)

            # Count active features.
            active = acts > 0
            feature_counts[layer] += (
                active.sum(dim=0).cpu().numpy()
            )

            # For each feature, find its strongest position
            # within this sequence.
            scores, positions = acts.max(dim=0)

            scores = scores.cpu().numpy()
            positions = positions.cpu().numpy()

            old_scores = feature_best_scores[layer]

            # Merge the sequence score with the stored top contexts.
            combined = np.concatenate(
                [old_scores, scores[:, None]], axis=1
            )

            top_indices = np.argpartition(
                combined,
                -TOP_CONTEXTS,
                axis=1
            )[:, -TOP_CONTEXTS:]

            new_scores = np.take_along_axis(
                combined, top_indices, axis=1
            )

            new_seq = np.where(
                top_indices == TOP_CONTEXTS,
                seq_idx,
                feature_best_seq[layer][
                    np.arange(8192)[:, None],
                    np.minimum(top_indices, TOP_CONTEXTS - 1)
                ]
            )

            # Rebuild selected sequence IDs safely.
            merged_seq = np.concatenate(
                [
                    feature_best_seq[layer],
                    np.full((8192, 1), seq_idx, dtype=np.int32)
                ],
                axis=1
            )

            merged_pos = np.concatenate(
                [
                    feature_best_pos[layer],
                    positions[:, None].astype(np.int32)
                ],
                axis=1
            )

            new_seq = np.take_along_axis(
                merged_seq, top_indices, axis=1
            )
            new_pos = np.take_along_axis(
                merged_pos, top_indices, axis=1
            )

            feature_best_scores[layer] = new_scores
            feature_best_seq[layer] = new_seq
            feature_best_pos[layer] = new_pos

            del sae, acts, residual

        del outputs, hidden_states
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

print("\nFeature extraction completed.")

# ------------------------- SELECT FEATURES -----------------------

print("\nSelecting active features and contexts...")

context_rows = []
candidate_rows = []

# Sentence embedding model.
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE
)

for layer in tqdm(range(1, 37)):

    counts = feature_counts[layer]

    selected_features = np.argsort(
        counts
    )[-TOP_FEATURES:][::-1]

    for feature_id in selected_features:

        scores = feature_best_scores[layer][feature_id]
        seq_ids = feature_best_seq[layer][feature_id]
        positions = feature_best_pos[layer][feature_id]

        examples = []

        for score, seq_id, pos in zip(
            scores, seq_ids, positions
        ):
            if seq_id < 0 or pos < 0:
                continue

            # Decode a local context around the peak token.
            token_sequence = tokens[seq_id]

            start = max(0, int(pos) - 20)
            end = min(SEQ_LEN, int(pos) + 21)

            context = tokenizer.decode(
                token_sequence[start:end],
                skip_special_tokens=True
            ).strip()

            if len(context) < 10:
                continue

            examples.append({
                "layer": layer,
                "feature_id": int(feature_id),
                "activation_score": float(score),
                "sequence_id": int(seq_id),
                "token_position": int(pos),
                "context": context
            })

        # Keep distinct contexts.
        unique = {}
        for item in sorted(
            examples,
            key=lambda x: x["activation_score"],
            reverse=True
        ):
            unique.setdefault(item["context"], item)

        examples = list(unique.values())[:TOP_CONTEXTS]

        for item in examples:
            context_rows.append(item)

        # Need enough examples for semantic clustering.
        if len(examples) < 6:
            candidate_rows.append({
                "layer": layer,
                "feature_id": int(feature_id),
                "activation_count": int(counts[feature_id]),
                "context_count": len(examples),
                "semantic_clusters": 0,
                "silhouette_score": np.nan,
                "polysemantic_candidate": False,
                "reason": "Insufficient distinct contexts"
            })
            continue

        texts = [x["context"] for x in examples]

        embeddings = embedder.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        # Compare two semantic clusters.
        km = KMeans(
            n_clusters=2,
            random_state=42,
            n_init=10
        )

        labels = km.fit_predict(embeddings)

        cluster_sizes = np.bincount(labels)

        if len(set(labels)) > 1 and min(cluster_sizes) >= MIN_CLUSTER_SIZE:
            silhouette = float(
                silhouette_score(embeddings, labels,
                                 metric="cosine")
            )
        else:
            silhouette = -1.0

        # Conservative candidate criterion:
        # two reasonably populated, distinguishable context groups.
        candidate = (
            min(cluster_sizes) >= MIN_CLUSTER_SIZE
            and silhouette >= 0.20
        )

        candidate_rows.append({
            "layer": layer,
            "feature_id": int(feature_id),
            "activation_count": int(counts[feature_id]),
            "context_count": len(examples),
            "semantic_clusters": 2,
            "cluster_0_size": int(cluster_sizes[0]),
            "cluster_1_size": int(cluster_sizes[1]),
            "silhouette_score": silhouette,
            "polysemantic_candidate": bool(candidate),
            "reason": (
                "Two distinct context groups"
                if candidate else
                "No sufficiently separated context groups"
            )
        })

# ------------------------- SAVE RESULTS ---------------------------

contexts_df = pd.DataFrame(context_rows)
candidates_df = pd.DataFrame(candidate_rows)

contexts_path = os.path.join(
    OUT, "feature_context_examples.csv"
)

candidates_path = os.path.join(
    OUT, "polysemanticity_candidates.csv"
)

contexts_df.to_csv(contexts_path, index=False)
candidates_df.to_csv(candidates_path, index=False)

# Summary by layer.
layer_summary = (
    candidates_df.groupby("layer")
    .agg(
        examined_features=("feature_id", "count"),
        candidate_features=("polysemantic_candidate", "sum"),
        mean_silhouette=("silhouette_score", "mean")
    )
    .reset_index()
)

layer_summary.to_csv(
    os.path.join(OUT, "polysemanticity_layer_summary.csv"),
    index=False
)

summary = {
    "model": MODEL_ID,
    "revision": REVISION,
    "corpus": "WikiText-103 token cache",
    "sequences": len(tokens),
    "tokens": int(tokens.numel()),
    "layers": 36,
    "features_examined": int(len(candidates_df)),
    "candidate_features": int(
        candidates_df["polysemantic_candidate"].sum()
    ),
    "candidate_rule": (
        "At least 3 contexts in each of two semantic clusters "
        "and cosine silhouette score >= 0.20"
    ),
    "warning": (
        "Automated semantic clustering identifies candidates only. "
        "Context diversity does not establish polysemanticity. "
        "Manual semantic inspection and independent validation "
        "are required."
    )
}

with open(
    os.path.join(OUT, "experiment_summary.json"), "w"
) as f:
    json.dump(summary, f, indent=2)

# ------------------------- DISPLAY --------------------------------

print("\n" + "=" * 65)
print("POLYSEMANTICITY DISCOVERY COMPLETE")
print("=" * 65)

print("Features examined:", len(candidates_df))
print(
    "Candidate features:",
    int(candidates_df["polysemantic_candidate"].sum())
)

print("\nCandidates by layer:")
display(layer_summary)

print("\nHighest-scoring candidate features:")
display(
    candidates_df[
        candidates_df.polysemantic_candidate
    ].sort_values(
        "silhouette_score", ascending=False
    ).head(30)
)

print("\nSaved outputs:")
print(contexts_path)
print(candidates_path)
print(os.path.join(OUT, "polysemanticity_layer_summary.csv"))
print(os.path.join(OUT, "experiment_summary.json"))


In [ ]:
# ==================================================
# RESTORE EXISTING SAE CHECKPOINTS FROM GOOGLE DRIVE
# ==================================================

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os
import io

FOLDER_ID = "123D1uNo6HVvJKAMqOx3fHBxEUVzm4Rql"
SAE_DIR = "/content/saes"

os.makedirs(SAE_DIR, exist_ok=True)

# Authenticate with the Google account that has access
auth.authenticate_user()

drive_service = build("drive", "v3")

print("Finding SAE checkpoints...")

response = drive_service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields="files(id, name, mimeType, size)",
    pageSize=100
).execute()

files = response.get("files", [])

checkpoints = [
    f for f in files
    if f["name"].startswith("sae_layer_")
    and f["name"].endswith(".pt")
]

print("Checkpoints found:", len(checkpoints))

assert len(checkpoints) == 36, (
    f"Expected 36 checkpoints, found {len(checkpoints)}. "
    "Check folder permissions and contents."
)

# Download all checkpoints
for file in sorted(checkpoints, key=lambda x: x["name"]):
    destination = os.path.join(SAE_DIR, file["name"])

    if os.path.exists(destination):
        continue

    request = drive_service.files().get_media(
        fileId=file["id"]
    )

    with io.FileIO(destination, "wb") as output:
        downloader = MediaIoBaseDownload(output, request)
        done = False

        while not done:
            _, done = downloader.next_chunk()

    print("Downloaded:", file["name"])

# Verify
downloaded = [
    f for f in os.listdir(SAE_DIR)
    if f.startswith("sae_layer_") and f.endswith(".pt")
]

assert len(downloaded) == 36

print("\nSUCCESS: All 36 SAE checkpoints restored.")
print("SAE_DIR =", SAE_DIR)

In [ ]:

# ================================================================
# EXPERIMENT 02: POLYSEMANTICITY DISCOVERY
# Qwen2.5-3B Base | 36 Top-K SAEs | Natural Language
# ================================================================

import os
import gc
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------

ROOT = "/content/drive/MyDrive/Mapping_Feature_Drift"

MODEL_ID = "Qwen/Qwen2.5-3B"
REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

SAE_DIR = "/content/saes"

TOKEN_CACHE = f"{ROOT}/data/wikitext103_tokens.pt"

OUTPUT_DIR = (
    f"{ROOT}/sae_research_100k_v2/"
    "experiment_02_polysemanticity"
)

SEQ_LEN = 256
MAX_SEQUENCES = 190
BATCH_SIZE = 2

INPUT_DIM = 2048
LATENT_DIM = 8192
TOP_K = 32

TOP_FEATURES_PER_LAYER = 12
TOP_CONTEXTS_PER_FEATURE = 20
CONTEXT_WINDOW = 15

DEVICE = "cuda"

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert torch.cuda.is_available(), "GPU is required."
assert os.path.isdir(SAE_DIR), f"SAE directory missing: {SAE_DIR}"

print("Device:", torch.cuda.get_device_name(0))


# ------------------------------------------------
# 2. INSTALL DEPENDENCIES
# ------------------------------------------------

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "scikit-learn"
    ])

    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

from transformers import AutoTokenizer, AutoModel


# ------------------------------------------------
# 3. LOAD TOKENIZED WIKITEXT
# ------------------------------------------------

# Restore or regenerate the WikiText-103 token cache
if not os.path.isfile(TOKEN_CACHE):
    print("Token cache not found. Rebuilding it from WikiText-103...")

    from datasets import load_dataset
    from transformers import AutoTokenizer
    import torch
    import os

    os.makedirs(os.path.dirname(TOKEN_CACHE), exist_ok=True)

    MODEL_ID = "Qwen/Qwen2.5-3B"
    REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=REVISION
    )

    dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="train"
    )

    text = "\n".join(
        item for item in dataset["text"] if item.strip()
    )

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False
    )["input_ids"]

    # Preserve the original experiment size: 48,640 tokens
    token_ids = token_ids[:48640]

    torch.save(torch.tensor(token_ids, dtype=torch.long), TOKEN_CACHE)

    print(f"Token cache rebuilt: {len(token_ids):,} tokens")
else:
    print("Existing token cache found.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

print("\nLoading cached WikiText tokens...")

token_ids = torch.load(
    TOKEN_CACHE,
    map_location="cpu",
    weights_only=False
)

usable = min(
    (len(token_ids) // SEQ_LEN) * SEQ_LEN,
    MAX_SEQUENCES * SEQ_LEN
)

token_ids = token_ids[:usable].reshape(-1, SEQ_LEN)

NUM_SEQUENCES = len(token_ids)
FLAT_TOKENS = token_ids.reshape(-1).tolist()

print("Sequences:", NUM_SEQUENCES)
print("Tokens:", usable)


# ------------------------------------------------
# 4. LOAD SAE CHECKPOINTS
# ------------------------------------------------

def get_tensor(state_dict, names, expected_shape=None):

    for name in names:
        for key, value in state_dict.items():

            if not torch.is_tensor(value):
                continue

            if key.lower().endswith(name.lower()):

                if (
                    expected_shape is None
                    or tuple(value.shape) == tuple(expected_shape)
                ):
                    return value

    return None


def load_sae(layer):

    path = f"{SAE_DIR}/sae_layer_{layer:02d}.pt"

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    if "state_dict" not in checkpoint:
        raise ValueError(
            f"state_dict missing in checkpoint: {path}"
        )

    state = checkpoint["state_dict"]

    W_enc = get_tensor(
        state,
        ["W_enc", "encoder.weight"]
    )

    b_enc = get_tensor(
        state,
        ["b_enc", "encoder.bias"]
    )

    mean = get_tensor(
        state,
        ["mean", "input_mean", "training_mean"]
    )

    if W_enc is None or b_enc is None or mean is None:
        raise ValueError(
            f"Missing SAE tensors in layer {layer}. "
            f"Available keys: {list(state.keys())}"
        )

    # Standardize encoder orientation.
    if W_enc.shape == (LATENT_DIM, INPUT_DIM):
        W_enc = W_enc.T

    assert W_enc.shape == (INPUT_DIM, LATENT_DIM), (
        f"Unexpected W_enc shape in layer {layer}: {W_enc.shape}"
    )

    assert b_enc.numel() == LATENT_DIM
    assert mean.numel() == INPUT_DIM

    result = {
        "W_enc": W_enc.to(
            device=DEVICE, dtype=torch.float16
        ).contiguous(),

        "b_enc": b_enc.reshape(-1).to(
            device=DEVICE, dtype=torch.float16
        ),

        "mean": mean.reshape(-1).to(
            device=DEVICE, dtype=torch.float16
        )
    }

    del checkpoint, state
    return result


print("\nLoading all 36 SAE checkpoints...")

saes = [
    load_sae(layer)
    for layer in tqdm(range(1, 37))
]

print("All 36 SAEs loaded successfully.")


# ------------------------------------------------
# 5. LOAD QWEN MODEL
# ------------------------------------------------

print("\nLoading Qwen2.5-3B Base...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
).to(DEVICE)

model.eval()

print("Model loaded successfully.")


# ------------------------------------------------
# 6. EXTRACT SAE ACTIVATIONS
# ------------------------------------------------

# For every layer, preserve the strongest 20 activation
# contexts for every latent feature.

K_CONTEXTS = TOP_CONTEXTS_PER_FEATURE

best_scores = [
    torch.zeros(
        (K_CONTEXTS, LATENT_DIM),
        dtype=torch.float32,
        device=DEVICE
    )
    for _ in range(36)
]

best_positions = [
    torch.full(
        (K_CONTEXTS, LATENT_DIM),
        -1,
        dtype=torch.long,
        device=DEVICE
    )
    for _ in range(36)
]

activation_counts = [
    torch.zeros(
        LATENT_DIM,
        dtype=torch.long,
        device=DEVICE
    )
    for _ in range(36)
]

print("\nExtracting residual-stream activations...")

with torch.inference_mode():

    for start in tqdm(
        range(0, NUM_SEQUENCES, BATCH_SIZE)
    ):

        batch = token_ids[
            start:start + BATCH_SIZE
        ].to(DEVICE)

        output = model(
            input_ids=batch,
            output_hidden_states=True,
            use_cache=False
        )

        for layer in range(36):

            # Block output for this transformer layer.
            x = output.hidden_states[layer + 1]

            x = x.reshape(-1, INPUT_DIM).to(
                dtype=torch.float16
            )

            sae = saes[layer]

            # Match the centered SAE training input.
            centered = x - sae["mean"]

            # SAE encoder.
            pre = centered @ sae["W_enc"] + sae["b_enc"]

            pre = torch.relu(pre)

            # Top-K sparse activation.
            values, indices = torch.topk(
                pre,
                TOP_K,
                dim=-1
            )

            # Count feature activations.
            activation_counts[layer] += torch.bincount(
                indices.reshape(-1),
                minlength=LATENT_DIM
            )

            # Sparse matrix for context retrieval.
            sparse = torch.zeros_like(pre)

            sparse.scatter_(
                1,
                indices,
                values
            )

            # Global token positions.
            global_positions = (
                start * SEQ_LEN
                + torch.arange(
                    batch.shape[0] * SEQ_LEN,
                    device=DEVICE
                )
            )

            # Best activations in the current batch.
            batch_scores, batch_positions = torch.topk(
                sparse,
                min(K_CONTEXTS, sparse.shape[0]),
                dim=0
            )

            batch_positions = global_positions[
                batch_positions
            ]

            # Merge with previously retained activations.
            merged_scores = torch.cat(
                [best_scores[layer], batch_scores],
                dim=0
            )

            merged_positions = torch.cat(
                [best_positions[layer], batch_positions],
                dim=0
            )

            new_scores, order = torch.topk(
                merged_scores,
                K_CONTEXTS,
                dim=0
            )

            new_positions = torch.gather(
                merged_positions,
                0,
                order
            )

            best_scores[layer] = new_scores
            best_positions[layer] = new_positions

        del output, batch

        if start % 20 == 0:
            torch.cuda.empty_cache()

print("\nActivation extraction completed.")


# ------------------------------------------------
# 7. RELEASE GPU MEMORY
# ------------------------------------------------

del model, saes

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------
# 8. RETRIEVE STRONGEST ACTIVATING CONTEXTS
# ------------------------------------------------

print("\nRetrieving feature contexts...")

feature_records = []
context_records = []

for layer in range(36):

    counts = activation_counts[layer].cpu().numpy()

    scores = best_scores[layer].cpu().numpy()
    positions = best_positions[layer].cpu().numpy()

    # Most frequently active features.
    candidates = np.argsort(counts)[::-1]

    candidates = [
        int(f)
        for f in candidates
        if counts[f] >= 10
    ][:TOP_FEATURES_PER_LAYER]

    for feature in candidates:

        contexts = []
        seen = set()

        for j in range(K_CONTEXTS):

            position = int(positions[j, feature])
            score = float(scores[j, feature])

            if position < 0 or score <= 0:
                continue

            left = max(0, position - CONTEXT_WINDOW)

            right = min(
                len(FLAT_TOKENS),
                position + CONTEXT_WINDOW + 1
            )

            context = tokenizer.decode(
                FLAT_TOKENS[left:right],
                skip_special_tokens=True
            ).strip()

            if not context or context in seen:
                continue

            seen.add(context)

            contexts.append({
                "layer": layer + 1,
                "feature": feature,
                "activation": score,
                "token_position": position,
                "context": context
            })

        if len(contexts) < 8:
            continue

        feature_records.append({
            "layer": layer + 1,
            "feature": feature,
            "activation_count": int(counts[feature]),
            "context_count": len(contexts)
        })

        context_records.extend(contexts)

features_df = pd.DataFrame(feature_records)
contexts_df = pd.DataFrame(context_records)

print("Selected features:", len(features_df))
print("Unique contexts:", len(contexts_df))


# ------------------------------------------------
# 9. SEMANTIC EMBEDDING MODEL
# ------------------------------------------------

print("\nLoading semantic embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)


# ------------------------------------------------
# 10. SEMANTIC CLUSTERING
# ------------------------------------------------

print("\nAnalysing semantic diversity of features...")

candidate_results = []

contexts_df["semantic_cluster"] = -1

for _, row in tqdm(
    features_df.iterrows(),
    total=len(features_df)
):

    layer = int(row["layer"])
    feature = int(row["feature"])

    subset = contexts_df[
        (contexts_df["layer"] == layer)
        & (contexts_df["feature"] == feature)
    ].copy()

    texts = subset["context"].tolist()

    if len(texts) < 8:
        continue

    embeddings = embedder.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=32
    )

    clustering = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    )

    labels = clustering.fit_predict(embeddings)

    cluster_sizes = np.bincount(
        labels,
        minlength=2
    )

    if min(cluster_sizes) < 3:
        continue

    silhouette = float(
        silhouette_score(
            embeddings,
            labels,
            metric="cosine"
        )
    )

    contexts_df.loc[
        subset.index,
        "semantic_cluster"
    ] = labels

    candidate_results.append({

        "layer": layer,
        "feature": feature,

        "activation_count": int(
            row["activation_count"]
        ),

        "context_count": len(texts),

        "cluster_0_size": int(cluster_sizes[0]),

        "cluster_1_size": int(cluster_sizes[1]),

        "silhouette_score": silhouette,

        "candidate_polysemantic": silhouette >= 0.20
    })


# ------------------------------------------------
# 11. SAVE EXPERIMENT RESULTS
# ------------------------------------------------

results_df = pd.DataFrame(candidate_results)

if not results_df.empty:

    results_df = results_df.sort_values(
        ["candidate_polysemantic", "silhouette_score"],
        ascending=[False, False]
    )

features_df.to_csv(
    f"{OUTPUT_DIR}/selected_features.csv",
    index=False
)

contexts_df.to_csv(
    f"{OUTPUT_DIR}/feature_contexts_and_clusters.csv",
    index=False
)

results_df.to_csv(
    f"{OUTPUT_DIR}/polysemanticity_candidates.csv",
    index=False
)

summary = {

    "experiment": "Natural Language Polysemanticity Discovery",

    "model": MODEL_ID,

    "revision": REVISION,

    "dataset": "WikiText-103 raw train",

    "sequences": int(NUM_SEQUENCES),

    "tokens": int(usable),

    "transformer_layers": 36,

    "sae_input_dim": INPUT_DIM,

    "sae_latent_dim": LATENT_DIM,

    "sae_top_k": TOP_K,

    "features_examined": int(len(features_df)),

    "contexts_retrieved": int(len(contexts_df)),

    "features_clustered": int(len(results_df)),

    "candidate_features": (
        int(results_df["candidate_polysemantic"].sum())
        if not results_df.empty else 0
    ),

    "candidate_threshold": 0.20,

    "interpretation": (
        "Semantic clusters identify candidate polysemantic "
        "features. They do not independently establish "
        "polysemanticity or causal functionality."
    )
}

with open(
    f"{OUTPUT_DIR}/summary.json",
    "w"
) as f:

    json.dump(summary, f, indent=2)


# ------------------------------------------------
# 12. DISPLAY RESULTS
# ------------------------------------------------

print("\n" + "=" * 65)
print("POLYSEMANTICITY DISCOVERY COMPLETED")
print("=" * 65)

print(json.dumps(summary, indent=2))

if not results_df.empty:

    print("\nTOP 20 CANDIDATE FEATURES")

    display(results_df.head(20))

    print("\nACTUAL ACTIVATING CONTEXTS")

    top_candidates = results_df[
        results_df["candidate_polysemantic"]
    ].head(5)

    for _, candidate in top_candidates.iterrows():

        layer = int(candidate["layer"])
        feature = int(candidate["feature"])

        print(
            f"\nLayer {layer} | Feature {feature} | "
            f"Silhouette: {candidate['silhouette_score']:.3f}"
        )

        subset = contexts_df[
            (contexts_df["layer"] == layer)
            & (contexts_df["feature"] == feature)
        ]

        for cluster in [0, 1]:

            print(f"\nSemantic cluster {cluster}:")

            examples = subset[
                subset["semantic_cluster"] == cluster
            ]

            for context in examples["context"].head(5):

                print(" -", context)

else:
    print(
        "No features passed the minimum clustering requirements. "
        "Inspect the context and feature extraction outputs."
    )

print("\nAll results saved to:")
print(OUTPUT_DIR)

print("\nEXPERIMENT 02 FINISHED.")


In [ ]:
%pip install torch torchvision torchaudio


In [ ]:
# =====================================================================
# EXPERIMENT 02B: DEEP POLYSEMANTICITY ANALYSIS
# Same 48,640 WikiText tokens | Same Qwen2.5-3B | Same 36 trained SAEs
#
# Improvements:
# - One strongest context per 256-token sequence and feature
# - Sequence-level activation prevalence
# - Ubiquitous-feature detection
# - 16 features per layer (up to 576 features)
# - Near-duplicate context filtering
# - KMeans comparison for k=2,3,4
# - Multiple initializations
# - Bootstrap clustering stability
# - Cluster-specific TF-IDF terms
# - Actual context exports and candidate ranking
# =====================================================================

import os
import gc
import json
import math
import warnings
import subprocess
import sys
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------

ROOT = "/content/drive/MyDrive/Mapping_Feature_Drift"

MODEL_ID = "Qwen/Qwen2.5-3B"
REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

SAE_DIR = f"{ROOT}/sae_research_100k_v2/saes"

TOKEN_CACHE = f"{ROOT}/data/wikitext103_tokens.pt"

BASE_OUTPUT = (
    f"{ROOT}/sae_research_100k_v2/"
    "experiment_02_polysemanticity"
)

OUTPUT_DIR = f"{BASE_OUTPUT}/deep_same_corpus"

SEQ_LEN = 256
MAX_SEQUENCES = 190
BATCH_SIZE = 2

INPUT_DIM = 2048
LATENT_DIM = 8192
TOP_K = 32

# Deeper analysis settings
FEATURES_PER_LAYER = 16
CONTEXTS_PER_FEATURE = 40
CONTEXT_WINDOW = 15

MIN_SEQUENCE_PREVALENCE = 0.05
MAX_SEQUENCE_PREVALENCE = 0.95
MIN_CONTEXTS = 12

CLUSTER_COUNTS = [2, 3, 4]
CLUSTER_SEEDS = [17, 37, 71]
BOOTSTRAP_RUNS = 6
BOOTSTRAP_FRACTION = 0.80

# Near-duplicate embedding cosine threshold
DUPLICATE_COSINE_THRESHOLD = 0.96

# Fallback gracefully if GPU is not available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(ROOT), f"Project folder missing: {ROOT}"
assert os.path.isdir(SAE_DIR), f"SAE folder missing: {SAE_DIR}"
assert os.path.isfile(TOKEN_CACHE), f"Token cache missing: {TOKEN_CACHE}"

print("=" * 70)
print("EXPERIMENT 02B: DEEP POLYSEMANTICITY ANALYSIS")
print("=" * 70)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU fallback")
print("SAE directory:", SAE_DIR)
print("Output:", OUTPUT_DIR)


# ---------------------------------------------------------------------
# 2. DEPENDENCIES
# ---------------------------------------------------------------------

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, adjusted_rand_score
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentence-transformers",
        "scikit-learn"
    ])

    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, adjusted_rand_score
    from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import AutoTokenizer, AutoModel


# ---------------------------------------------------------------------
# 3. LOAD THE EXACT SAME TOKEN SAMPLE
# ---------------------------------------------------------------------

print("\n[1/9] Loading the existing WikiText token cache...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

token_ids = torch.load(
    TOKEN_CACHE,
    map_location="cpu",
    weights_only=False
)

usable = min(
    (len(token_ids) // SEQ_LEN) * SEQ_LEN,
    MAX_SEQUENCES * SEQ_LEN
)

token_ids = token_ids[:usable].reshape(-1, SEQ_LEN)

NUM_SEQUENCES = len(token_ids)

assert NUM_SEQUENCES == 190, (
    f"Expected 190 sequences, found {NUM_SEQUENCES}"
)

assert usable == 48640, (
    f"Expected 48,640 tokens, found {usable}"
)

print("Sequences:", NUM_SEQUENCES)
print("Tokens:", usable)


# ---------------------------------------------------------------------
# 4. LOAD THE EXISTING 36 SAEs
# ---------------------------------------------------------------------

print("\n[2/9] Loading all 36 existing SAE checkpoints...")


def get_tensor(state_dict, names):

    for name in names:
        for key, value in state_dict.items():

            if not torch.is_tensor(value):
                continue

            if key.lower().endswith(name.lower()):
                return value

    return None


def load_sae(layer):

    path = f"{SAE_DIR}/sae_layer_{layer:02d}.pt"

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = checkpoint["state_dict"]

    W_enc = get_tensor(
        state,
        ["W_enc", "encoder.weight"]
    )

    b_enc = get_tensor(
        state,
        ["b_enc", "encoder.bias"]
    )

    mean = get_tensor(
        state,
        ["mean", "input_mean", "training_mean"]
    )

    if W_enc is None or b_enc is None or mean is None:
        raise ValueError(
            f"Missing SAE tensors in layer {layer}. "
            f"Checkpoint keys: {list(state.keys())}"
        )

    if W_enc.shape == (LATENT_DIM, INPUT_DIM):
        W_enc = W_enc.T

    assert W_enc.shape == (INPUT_DIM, LATENT_DIM), (
        f"Unexpected encoder shape in layer {layer}: {W_enc.shape}"
    )

    assert b_enc.numel() == LATENT_DIM
    assert mean.numel() == INPUT_DIM

    result = {
        "W_enc": W_enc.to(
            DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        ).contiguous(),

        "b_enc": b_enc.reshape(-1).to(
            DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        ),

        "mean": mean.reshape(-1).to(
            DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        )
    }

    del checkpoint, state, W_enc, b_enc, mean

    return result

saes = [
    load_sae(layer)
    for layer in tqdm(range(1, 37))
]

print("Loaded all 36 SAEs.")


# ---------------------------------------------------------------------
# 5. LOAD QWEN
# ---------------------------------------------------------------------

print("\n[3/9] Loading Qwen2.5-3B Base...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    attn_implementation="sdpa" if DEVICE == "cuda" else "eager"
).to(DEVICE)

model.eval()

print("Model loaded.")


# ---------------------------------------------------------------------
# 6. SEQUENCE-LEVEL FEATURE ACTIVATION ANALYSIS
# ---------------------------------------------------------------------

print("\n[4/9] Extracting sequence-level feature activations...")

activation_counts = [
    torch.zeros(
        LATENT_DIM,
        dtype=torch.long,
        device=DEVICE
    )
    for _ in range(36)
]

sequence_presence = [
    torch.zeros(
        NUM_SEQUENCES,
        LATENT_DIM,
        dtype=torch.bool,
        device=DEVICE
    )
    for _ in range(36)
]

sequence_best_scores = [
    torch.zeros(
        NUM_SEQUENCES,
        LATENT_DIM,
        dtype=torch.float32,
        device=DEVICE
    )
    for _ in range(36)
]

sequence_best_positions = [
    torch.full(
        (NUM_SEQUENCES, LATENT_DIM),
        -1,
        dtype=torch.int16,
        device=DEVICE
    )
    for _ in range(36)
]


with torch.inference_mode():

    for start in tqdm(
        range(0, NUM_SEQUENCES, BATCH_SIZE)
    ):

        batch = token_ids[
            start:start + BATCH_SIZE
        ].to(DEVICE)

        output = model(
            input_ids=batch,
            output_hidden_states=True,
            use_cache=False
        )

        for layer in range(36):

            x = output.hidden_states[layer + 1]

            x = x.reshape(
                batch.shape[0],
                SEQ_LEN,
                INPUT_DIM
            )

            sae = saes[layer]

            centered = x - sae["mean"]

            pre = (
                centered.reshape(-1, INPUT_DIM)
                @ sae["W_enc"]
            ) + sae["b_enc"]

            pre = torch.relu(pre)

            values, indices = torch.topk(
                pre,
                TOP_K,
                dim=-1
            )

            indices = indices.reshape(
                batch.shape[0],
                SEQ_LEN,
                TOP_K
            )

            values = values.reshape(
                batch.shape[0],
                SEQ_LEN,
                TOP_K
            )

            # Count all Top-K activations.
            activation_counts[layer] += torch.bincount(
                indices.reshape(-1),
                minlength=LATENT_DIM
            )

            # Process each sequence independently.
            for b in range(batch.shape[0]):

                seq_id = start + b

                idx = indices[b].reshape(-1)

                vals = values[b].float().reshape(-1)

                token_positions = (
                    torch.arange(
                        SEQ_LEN,
                        device=DEVICE
                    )
                    .unsqueeze(1)
                    .expand(SEQ_LEN, TOP_K)
                    .reshape(-1)
                )

                counts = torch.bincount(
                    idx,
                    minlength=LATENT_DIM
                )

                present = counts > 0

                sequence_presence[layer][seq_id] = present

                # Maximum activation per feature in this sequence.
                max_values = torch.full(
                    (LATENT_DIM,),
                    -float("inf"),
                    dtype=torch.float32,
                    device=DEVICE
                )

                max_values.scatter_reduce_(
                    0,
                    idx,
                    vals,
                    reduce="amax",
                    include_self=True
                )

                # Find the earliest token position attaining that maximum.
                is_max = (
                    vals >= max_values[idx] - 1e-7
                )

                best_positions = torch.full(
                    (LATENT_DIM,),
                    SEQ_LEN,
                    dtype=torch.long,
                    device=DEVICE
                )

                best_positions.scatter_reduce_(
                    0,
                    idx[is_max],
                    token_positions[is_max],
                    reduce="amin",
                    include_self=True
                )

                max_values[~present] = 0

                best_positions[~present] = -1

                sequence_best_scores[layer][seq_id] = max_values

                sequence_best_positions[layer][seq_id] = (
                    best_positions.to(torch.int16)
                )

        del output, batch, pre

        if start % 20 == 0 and DEVICE == "cuda":
            torch.cuda.empty_cache()


print("Sequence-level activation extraction complete.")


# ---------------------------------------------------------------------
# 7. RELEASE MODEL AND SAE MEMORY
# ---------------------------------------------------------------------

del model, saes

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------
# 8. SELECT FEATURES AND RETRIEVE DIVERSE CONTEXTS
# ---------------------------------------------------------------------

print("\n[5/9] Selecting features and retrieving contexts...")

feature_records = []
context_records = []
ubiquitous_records = []

for layer in range(36):

    counts = activation_counts[layer].cpu().numpy()

    presence = (
        sequence_presence[layer]
        .sum(dim=0)
        .cpu()
        .numpy()
    )

    prevalence = presence / NUM_SEQUENCES

    # Flag features that appear in nearly every sequence.
    ubiquitous_ids = np.where(
        prevalence >= MAX_SEQUENCE_PREVALENCE
    )[0]

    for feature in ubiquitous_ids:

        ubiquitous_records.append({
            "layer": layer + 1,
            "feature": int(feature),
            "activation_count": int(counts[feature]),
            "sequence_count": int(presence[feature]),
            "sequence_prevalence": float(prevalence[feature])
        })

    # Primary discovery excludes near-ubiquitous features.
    eligible = np.where(
        (prevalence >= MIN_SEQUENCE_PREVALENCE)
        & (prevalence < MAX_SEQUENCE_PREVALENCE)
        & (counts >= 20)
    )[0]

    if len(eligible) == 0:
        continue

    # Rank by number of sequences in which the feature activates.
    ranked = eligible[
        np.argsort(presence[eligible])[::-1]
    ]

    selected = ranked[:FEATURES_PER_LAYER]

    score_matrix = (
        sequence_best_scores[layer][:, selected]
        .cpu()
        .numpy()
    )

    position_matrix = (
        sequence_best_positions[layer][:, selected]
        .cpu()
        .numpy()
    )

    for col, feature in enumerate(selected):

        scores = score_matrix[:, col]
        positions = position_matrix[:, col]

        valid = np.where(
            (scores > 0) & (positions >= 0)
        )[0]

        if len(valid) < MIN_CONTEXTS:
            continue

        # Select strongest activation from distinct sequences.
        ranked_sequences = valid[
            np.argsort(scores[valid])[::-1]
        ]

        chosen = ranked_sequences[:CONTEXTS_PER_FEATURE]

        contexts = []

        for seq_id in chosen:

            local_position = int(positions[seq_id])

            seq_tokens = token_ids[seq_id].tolist()

            left = max(
                0,
                local_position - CONTEXT_WINDOW
            )

            right = min(
                SEQ_LEN,
                local_position + CONTEXT_WINDOW + 1
            )

            context = tokenizer.decode(
                seq_tokens[left:right],
                skip_special_tokens=True
            ).strip()

            active_token = tokenizer.decode(
                [seq_tokens[local_position]],
                skip_special_tokens=True
            )

            if not context:
                continue

            contexts.append({
                "layer": layer + 1,
                "feature": int(feature),
                "sequence_id": int(seq_id),
                "token_position": local_position,
                "activation": float(scores[seq_id]),
                "active_token": active_token,
                "context": context
            })

        if len(contexts) < MIN_CONTEXTS:
            continue

        feature_records.append({
            "layer": layer + 1,
            "feature": int(feature),
            "activation_count": int(counts[feature]),
            "active_sequence_count": int(presence[feature]),
            "sequence_prevalence": float(prevalence[feature]),
            "retrieved_contexts": len(contexts)
        })

        context_records.extend(contexts)


features_df = pd.DataFrame(feature_records)
contexts_df = pd.DataFrame(context_records)
ubiquitous_df = pd.DataFrame(ubiquitous_records)

print("Selected features:", len(features_df))
print("Retrieved contexts:", len(contexts_df))
print("Ubiquitous features flagged:", len(ubiquitous_df))

assert len(features_df) > 0, "No eligible features found."


# ---------------------------------------------------------------------
# 9. SEMANTIC EMBEDDINGS
# ---------------------------------------------------------------------

print("\n[6/9] Loading sentence embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE
)

print("Embedding contexts...")

all_texts = contexts_df["context"].tolist()

all_embeddings = embedder.encode(
    all_texts,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

contexts_df["embedding_row"] = np.arange(len(contexts_df))


# ---------------------------------------------------------------------
# 10. CONTEXT DEDUPLICATION AND CLUSTERING
# ---------------------------------------------------------------------

print("\n[7/9] Running semantic diversity and stability analysis...")


def deduplicate_contexts(subset, embeddings):

    # Input contexts are sorted by activation strength.
    # Keep the strongest context and reject near duplicates.
    kept_indices = []

    for i in range(len(embeddings)):

        if not kept_indices:
            kept_indices.append(i)
            continue

        similarities = (
            embeddings[kept_indices] @ embeddings[i]
        )

        if np.max(similarities) < DUPLICATE_COSINE_THRESHOLD:
            kept_indices.append(i)

    return kept_indices


def cluster_terms(texts, labels, cluster_id):

    selected_texts = [
        texts[i]
        for i in range(len(texts))
        if labels[i] == cluster_id
    ]

    if len(selected_texts) < 2:
        return ""

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=100,
            ngram_range=(1, 2),
            min_df=1
        )

        matrix = vectorizer.fit_transform(selected_texts)

        terms = np.asarray(
            vectorizer.get_feature_names_out()
        )

        weights = np.asarray(
            matrix.mean(axis=0)
        ).ravel()

        top = np.argsort(weights)[::-1][:8]

        return ", ".join(terms[top])

    except ValueError:
        return ""


def evaluate_feature(texts, embeddings):

    n = len(texts)

    if n < MIN_CONTEXTS:
        return None

    best = None

    # Compare multiple cluster counts and random initializations.
    for k in CLUSTER_COUNTS:

        if n < k * 4:
            continue

        runs = []

        for seed in CLUSTER_SEEDS:

            model_k = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=10
            )

            labels = model_k.fit_predict(embeddings)

            sizes = np.bincount(
                labels,
                minlength=k
            )

            # Avoid tiny clusters being treated as separate meanings.
            if sizes.min() < 4:
                continue

            sil = float(
                silhouette_score(
                    embeddings,
                    labels,
                    metric="cosine"
                )
            )

            runs.append({
                "labels": labels,
                "silhouette": sil,
                "sizes": sizes
            })

        if not runs:
            continue

        # Choose the strongest run for this k.
        best_run = max(
            runs,
            key=lambda r: r["silhouette"]
        )

        # Measure agreement across independent initializations.
        ari_values = [
            adjusted_rand_score(a["labels"], b["labels"])
            for a, b in combinations(runs, 2)
        ]

        seed_stability = (
            float(np.mean(ari_values))
            if ari_values else 1.0
        )

        candidate = {
            "k": k,
            "labels": best_run["labels"],
            "silhouette": best_run["silhouette"],
            "sizes": best_run["sizes"],
            "seed_stability": seed_stability
        }

        if (
            best is None
            or candidate["silhouette"] > best["silhouette"]
        ):
            best = candidate

    if best is None:
        return None

    labels = best["labels"]
    k = best["k"]

    # Bootstrap stability:
    # fit on 80% of contexts and predict all contexts.
    bootstrap_ari = []

    rng = np.random.default_rng(2026)

    sample_size = max(
        k * 4,
        int(n * BOOTSTRAP_FRACTION)
    )

    for b in range(BOOTSTRAP_RUNS):

        sample_idx = rng.choice(
            n,
            size=min(sample_size, n),
            replace=False
        )

        bootstrap_model = KMeans(
            n_clusters=k,
            random_state=1000 + b,
            n_init=10
        )

        bootstrap_model.fit(
            embeddings[sample_idx]
        )

        predicted = bootstrap_model.predict(
            embeddings
        )

        bootstrap_ari.append(
            adjusted_rand_score(labels, predicted)
        )

    bootstrap_stability = float(
        np.mean(bootstrap_ari)
    )

    min_cluster_size = int(best["sizes"].min())

    max_cluster_fraction = float(
        best["sizes"].max() / n
    )

    terms = {}

    for cluster_id in range(k):
        terms[cluster_id] = cluster_terms(
            texts,
            labels,
            cluster_id
        )

    # Conservative discovery tiers.
    if (
        best["silhouette"] >= 0.25
        and bootstrap_stability >= 0.75
        and seed_stability >= 0.70
        and min_cluster_size >= 5
        and max_cluster_fraction <= 0.85
        and n >= 16
    ):
        tier = "strong_candidate"

    elif (
        best["silhouette"] >= 0.20
        and bootstrap_stability >= 0.60
        and seed_stability >= 0.50
        and min_cluster_size >= 4
    ):
        tier = "provisional_candidate"

    else:
        tier = "not_supported"

    return {
        "k": k,
        "labels": labels,
        "silhouette": float(best["silhouette"]),
        "seed_stability": seed_stability,
        "bootstrap_stability": bootstrap_stability,
        "min_cluster_size": min_cluster_size,
        "max_cluster_fraction": max_cluster_fraction,
        "cluster_sizes": best["sizes"].tolist(),
        "cluster_terms": terms,
        "tier": tier
    }


metric_records = []
deep_context_records = []

grouped = contexts_df.groupby(
    ["layer", "feature"],
    sort=False
)

for (layer, feature), subset in tqdm(
    grouped,
    total=grouped.ngroups
):

    subset = subset.sort_values(
        "activation",
        ascending=False
    ).copy()

    original_rows = subset.index.to_numpy()

    embedding_rows = subset[
        "embedding_row"
    ].to_numpy(dtype=int)

    embeddings = all_embeddings[embedding_rows]

    # Deduplicate near-identical contexts.
    keep = deduplicate_contexts(
        subset,
        embeddings
    )

    subset = subset.iloc[keep].copy()
    embeddings = embeddings[keep]

    texts = subset["context"].tolist()

    if len(texts) < MIN_CONTEXTS:
        continue

    result = evaluate_feature(
        texts,
        embeddings
    )

    if result is None:
        continue

    labels = result.pop("labels")

    terms = result.pop("cluster_terms")

    sizes = result.pop("cluster_sizes")

    for cluster_id in range(result["k"]):

        result[f"cluster_{cluster_id}_size"] = int(
            sizes[cluster_id]
        )

        result[f"cluster_{cluster_id}_terms"] = terms[cluster_id]

    result["layer"] = int(layer)
    result["feature"] = int(feature)
    result["contexts_before_dedup"] = int(len(original_rows))
    result["contexts_after_dedup"] = int(len(subset))
    result["distinct_sequences"] = int(
        subset["sequence_id"].nunique()
    )

    metric_records.append(result)

    subset["semantic_cluster"] = labels

    for column in [
        "layer",
        "feature",
        "sequence_id",
        "token_position",
        "activation",
        "active_token",
        "context",
        "semantic_cluster"
    ]:
        if column not in subset.columns:
            subset[column] = np.nan

    deep_context_records.extend(
        subset[
            [
                "layer",
                "feature",
                "sequence_id",
                "token_position",
                "activation",
                "active_token",
                "context",
                "semantic_cluster"
            ]
        ].to_dict("records")
    )


metrics_df = pd.DataFrame(metric_records)
deep_contexts_df = pd.DataFrame(deep_context_records)


# ---------------------------------------------------------------------
# 11. MERGE WITH FIRST-PASS CANDIDATES
# ---------------------------------------------------------------------

print("\n[8/9] Comparing with the original discovery results...")

old_candidates_path = (
    f"{BASE_OUTPUT}/polysemanticity_candidates.csv"
)

if os.path.exists(old_candidates_path):

    old_df = pd.read_csv(old_candidates_path)

    old_ids = set(
        zip(
            old_df["layer"].astype(int),
            old_df["feature"].astype(int)
        )
    )

    metrics_df["present_in_first_pass"] = [
        (int(row.layer), int(row.feature)) in old_ids
        for row in metrics_df.itertuples()
    ]

else:
    metrics_df["present_in_first_pass"] = False


# Rank candidates by evidence quality.
tier_order = {
    "strong_candidate": 0,
    "provisional_candidate": 1,
    "not_supported": 2
}

metrics_df["tier_rank"] = metrics_df["tier"].map(
    tier_order
)

metrics_df = metrics_df.sort_values(
    [
        "tier_rank",
        "bootstrap_stability",
        "silhouette",
        "seed_stability",
        "distinct_sequences"
    ],
    ascending=[True, False, False, False, False]
).reset_index(drop=True)


# ---------------------------------------------------------------------
# 12. LAYER-WISE SUMMARY
# ---------------------------------------------------------------------

if not metrics_df.empty:

    layer_summary = (
        metrics_df.groupby("layer")
        .agg(
            features_tested=("feature", "count"),
            strong_candidates=(
                "tier",
                lambda x: int((x == "strong_candidate").sum())
            ),
            provisional_candidates=(
                "tier",
                lambda x: int((x == "provisional_candidate").sum())
            ),
            mean_silhouette=("silhouette", "mean"),
            mean_bootstrap_stability=(
                "bootstrap_stability", "mean"
            ),
            mean_contexts=("contexts_after_dedup", "mean")
        )
        .reset_index()
    )

else:
    layer_summary = pd.DataFrame()


# ---------------------------------------------------------------------
# 13. SAVE ALL OUTPUTS
# ---------------------------------------------------------------------

print("\n[9/9] Saving deep analysis results...")

features_df.to_csv(
    f"{OUTPUT_DIR}/features_selected.csv",
    index=False
)

contexts_df.to_csv(
    f"{OUTPUT_DIR}/sequence_level_contexts_before_clustering.csv",
    index=False
)

deep_contexts_df.to_csv(
    f"{OUTPUT_DIR}/diverse_contexts_with_clusters.csv",
    index=False
)

metrics_df.to_csv(
    f"{OUTPUT_DIR}/deep_polysemanticity_metrics.csv",
    index=False
)

ubiquitous_df.to_csv(
    f"{OUTPUT_DIR}/ubiquitous_features.csv",
    index=False
)

layer_summary.to_csv(
    f"{OUTPUT_DIR}/layer_summary.csv",
    index=False
)

summary = {
    "experiment": "Deep Natural Language Polysemanticity Analysis",
    "model": MODEL_ID,
    "revision": REVISION,
    "dataset": "Same WikiText-103 token cache as first pass",
    "sequences": int(NUM_SEQUENCES),
    "tokens": int(usable),
    "transformer_layers": 36,
    "sae_input_dim": INPUT_DIM,
    "sae_latent_dim": LATENT_DIM,
    "sae_top_k": TOP_K,
    "features_per_layer": FEATURES_PER_LAYER,
    "feature_context_policy": (
        "One strongest activation context per 256-token sequence"
    ),
    "near_duplicate_cosine_threshold": DUPLICATE_COSINE_THRESHOLD,
    "cluster_counts_tested": CLUSTER_COUNTS,
    "cluster_seeds": CLUSTER_SEEDS,
    "bootstrap_runs": BOOTSTRAP_RUNS,
    "features_selected": int(len(features_df)),
    "features_clustered": int(len(metrics_df)),
    "contexts_retrieved": int(len(contexts_df)),
    "contexts_after_clustering": int(len(deep_contexts_df)),
    "ubiquitous_features_flagged": int(len(ubiquitous_df)),
    "strong_candidates": (
        int((metrics_df["tier"] == "strong_candidate").sum())
        if not metrics_df.empty else 0
    ),
    "provisional_candidates": (
        int((metrics_df["tier"] == "provisional_candidate").sum())
        if not metrics_df.empty else 0
    ),
    "not_supported": (
        int((metrics_df["tier"] == "not_supported").sum())
        if not metrics_df.empty else 0
    ),
    "important_limitation": (
        "The 256-token sequences are chunks of the same token stream, "
        "not guaranteed independent documents. Candidate status is "
        "exploratory and requires semantic/manual validation."
    ),
    "interpretation": (
        "Strong and provisional candidates show stable semantic "
        "separation in retrieved contexts. This is not proof of "
        "independent causal functions."
    )
}

with open(
    f"{OUTPUT_DIR}/deep_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)


# ---------------------------------------------------------------------
# 14. DISPLAY RESULTS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("DEEP POLYSEMANTICITY ANALYSIS COMPLETE")
print("=" * 75)

print(json.dumps(summary, indent=2))

if not metrics_df.empty:

    print("\nTOP 25 FEATURES")

    display(
        metrics_df[
            [
                "layer",
                "feature",
                "tier",
                "contexts_after_dedup",
                "silhouette",
                "seed_stability",
                "bootstrap_stability",
                "cluster_sizes"
            ]
        ].head(25)
    )

    print("\nLAYER-WISE RESULTS")

    display(layer_summary)

    print("\nTOP CANDIDATE CONTEXTS")

    top_candidates = metrics_df[
        metrics_df["tier"].isin(
            ["strong_candidate", "provisional_candidate"]
        )
    ].head(8)

    for row in top_candidates.itertuples():

        print("\n" + "-" * 70)
        print(
            f"Layer {row.layer} | Feature {row.feature} | "
            f"Tier: {row.tier}"
        )

        print(
            f"Silhouette: {row.silhouette:.3f} | "
            f"Bootstrap stability: {row.bootstrap_stability:.3f} | "
            f"Seed stability: {row.seed_stability:.3f}"
        )

        subset = deep_contexts_df[
            (deep_contexts_df["layer"] == row.layer)
            & (deep_contexts_df["feature"] == row.feature)
        ]

        for cluster in sorted(
            subset["semantic_cluster"].unique()
        ):

            print(
                f"\nCluster {int(cluster)}:"
            )

            examples = subset[
                subset["semantic_cluster"] == cluster
            ].sort_values(
                "activation",
                ascending=False
            )

            for context in examples["context"].head(6):
                print(" -", context)

else:
    print("No features passed clustering requirements.")


print("\nAll results saved to:")
print(OUTPUT_DIR)

print("\nEXPERIMENT 02B FINISHED.")


In [ ]:

# =============================================================================
# EXPERIMENT 03: RIGOROUS SAE POLYSEMANTICITY VALIDATION
#
# Reuses:
#   - Qwen2.5-3B Base (same revision)
#   - Existing 36 trained SAEs
#   - Experiment 02B's 79 candidate features
#
# Tests:
#   1. Independent-document replication on WikiText-103 TEST
#   2. Cross-corpus replication on C4 English VALIDATION
#   3. Matched non-candidate SAE feature controls
#   4. Semantic clustering, permutation significance, bootstrap stability
#   5. Document-level context exports and ranked evidence
#
# IMPORTANT:
#   This is a replication and semantic-validation experiment.
#   It does not by itself establish causal polysemanticity.
# =============================================================================

import os, sys, gc, json, math, random, warnings, subprocess
from pathlib import Path
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings("ignore")

# --------------------------- DEPENDENCIES ------------------------------------

def install_if_missing(package, import_name=None):
    import importlib.util
    if importlib.util.find_spec(import_name or package) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

install_if_missing("datasets")
install_if_missing("sentence-transformers", "sentence_transformers")
install_if_missing("scikit-learn", "sklearn")

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import TfidfVectorizer

# --------------------------- CONFIGURATION -----------------------------------

ROOT = "/content/drive/MyDrive/Mapping_Feature_Drift"
V2 = f"{ROOT}/sae_research_100k_v2"

SAE_DIR = f"{V2}/saes"
OLD_OUTPUT = f"{V2}/experiment_02_polysemanticity/deep_same_corpus"

MODEL_ID = "Qwen/Qwen2.5-3B"
REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

CANDIDATE_FILE = f"{OLD_OUTPUT}/deep_polysemanticity_metrics.csv"
FEATURE_FILE = f"{OLD_OUTPUT}/features_selected.csv"

SEQ_LEN = 256
MAX_DOCS_WIKITEXT = 500
MAX_DOCS_C4 = 500
MAX_TOKENS_PER_DOCUMENT = 768

CONTEXT_WINDOW = 18
MIN_DOCUMENTS = 12

CLUSTER_COUNTS = [2, 3, 4]
SEEDS = [17, 37, 71]
BOOTSTRAP_RUNS = 6
PERMUTATION_RUNS = 199

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# A timestamped directory protects previous experiment outputs.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT = f"{V2}/experiment_03_polysemanticity_validation/run_{RUN_ID}"
os.makedirs(OUTPUT, exist_ok=True)

print("=" * 75)
print("EXPERIMENT 03: RIGOROUS POLYSEMANTICITY VALIDATION")
print("=" * 75)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT)

# --------------------------- VERIFY INPUTS -----------------------------------

assert os.path.isdir(ROOT), f"Project folder missing: {ROOT}"
assert os.path.isdir(SAE_DIR), f"SAE directory missing: {SAE_DIR}"
assert os.path.isfile(CANDIDATE_FILE), f"Candidate file missing: {CANDIDATE_FILE}"
assert os.path.isfile(FEATURE_FILE), f"Feature file missing: {FEATURE_FILE}"

checkpoint_files = [
    f for f in os.listdir(SAE_DIR)
    if f.startswith("sae_layer_") and f.endswith(".pt")
]
assert len(checkpoint_files) == 36, (
    f"Expected 36 SAE checkpoints, found {len(checkpoint_files)}"
)

# --------------------------- LOAD CANDIDATES ---------------------------------

metrics_old = pd.read_csv(CANDIDATE_FILE)
selected_old = pd.read_csv(FEATURE_FILE)

candidate_df = metrics_old[
    metrics_old["tier"].isin(
        ["strong_candidate", "provisional_candidate"]
    )
].copy()

# Normalize feature identifiers
candidate_df["layer"] = candidate_df["layer"].astype(int)
candidate_df["feature"] = candidate_df["feature"].astype(int)

selected_old["layer"] = selected_old["layer"].astype(int)
selected_old["feature"] = selected_old["feature"].astype(int)

# Retrieve sequence prevalence from the correct source file
prevalence_data = selected_old[
    ["layer", "feature", "sequence_prevalence"]
].drop_duplicates(["layer", "feature"])

candidate_df = candidate_df.merge(
    prevalence_data,
    on=["layer", "feature"],
    how="left",
    validate="one_to_one"
)

assert candidate_df["sequence_prevalence"].notna().all(), (
    "Some candidate features have no matching sequence prevalence."
)

candidate_df = candidate_df.drop_duplicates(["layer", "feature"])

assert len(candidate_df) == 79, (
    f"Expected 79 candidates, found {len(candidate_df)}"
)

print("Candidates:", len(candidate_df))
print("Prevalence successfully matched:", candidate_df["sequence_prevalence"].notna().sum())

assert len(candidate_df) > 0, "No candidate features found."

print("\nCandidates:", len(candidate_df))
print("Strong:", (candidate_df["tier"] == "strong_candidate").sum())
print("Provisional:", (candidate_df["tier"] == "provisional_candidate").sum())

# --------------------------- MATCHED CONTROLS -------------------------------

# Match one non-candidate feature per candidate, in the same layer,
# using the closest sequence prevalence from Experiment 02B.

control_records = []
candidate_pairs = set(zip(candidate_df.layer, candidate_df.feature))

for layer in sorted(candidate_df.layer.unique()):
    candidates_layer = candidate_df[candidate_df.layer == layer]
    available = selected_old[
        selected_old["layer"].astype(int) == int(layer)
    ].copy()

    available["feature"] = available["feature"].astype(int)
    available = available[
        ~available["feature"].isin(
            candidates_layer["feature"].tolist()
        )
    ]

    used = set()

    for row in candidates_layer.itertuples():
        if available.empty:
            continue

        target_prev = float(row.sequence_prevalence)

        available["distance"] = (
            available["sequence_prevalence"] - target_prev
        ).abs()

        available = available[
            ~available["feature"].isin(used)
        ]

        if available.empty:
            continue

        chosen = available.sort_values("distance").iloc[0]
        control_feature = int(chosen["feature"])
        used.add(control_feature)

        control_records.append({
            "layer": int(layer),
            "feature": control_feature,
            "matched_candidate": int(row.feature),
            "candidate_tier": row.tier,
            "prevalence_distance": float(chosen["distance"]),
            "role": "matched_control"
        })

controls_df = pd.DataFrame(control_records)

if controls_df.empty:
    raise RuntimeError("Could not construct matched control features.")

print("Matched controls:", len(controls_df))

# Candidate and control feature identifiers by layer.
targets = defaultdict(list)

for row in candidate_df.itertuples():
    targets[int(row.layer)].append({
        "feature": int(row.feature),
        "role": "candidate",
        "tier": row.tier
    })

for row in controls_df.itertuples():
    targets[int(row.layer)].append({
        "feature": int(row.feature),
        "role": "matched_control",
        "tier": row.candidate_tier
    })

# --------------------------- LOAD MODEL --------------------------------------

print("\nLoading tokenizer and Qwen model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=DTYPE,
    attn_implementation="sdpa" if DEVICE == "cuda" else "eager"
).to(DEVICE)

model.eval()

# --------------------------- LOAD REQUIRED SAEs ------------------------------

INPUT_DIM = 2048
LATENT_DIM = 8192
TOP_K = 32

def get_tensor(state, names):
    for name in names:
        for key, value in state.items():
            if torch.is_tensor(value) and key.lower().endswith(name.lower()):
                return value
    return None

saes = {}

print("\nLoading SAEs for candidate/control layers...")

for layer in tqdm(sorted(targets.keys())):
    path = f"{SAE_DIR}/sae_layer_{layer:02d}.pt"

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )

    state = checkpoint["state_dict"]

    W = get_tensor(state, ["W_enc", "encoder.weight"])
    b = get_tensor(state, ["b_enc", "encoder.bias"])
    mean = get_tensor(state, ["mean", "input_mean", "training_mean"])

    if W is None or b is None or mean is None:
        raise ValueError(f"Missing encoder tensors in layer {layer}")

    if W.shape == (LATENT_DIM, INPUT_DIM):
        W = W.T

    assert W.shape == (INPUT_DIM, LATENT_DIM), (
        f"Incorrect W_enc shape at layer {layer}: {W.shape}"
    )

    saes[layer] = {
        "W": W.to(DEVICE, dtype=DTYPE).contiguous(),
        "b": b.reshape(-1).to(DEVICE, dtype=DTYPE),
        "mean": mean.reshape(-1).to(DEVICE, dtype=DTYPE)
    }

    del checkpoint, state, W, b, mean
    gc.collect()

print("SAEs loaded:", len(saes))

# --------------------------- CONTEXT EXTRACTION -----------------------------

# One best activation context per feature per document.
# This prevents neighboring 256-token chunks from being counted
# as independent examples.

def extract_document(doc_id, text, corpus):
    if not isinstance(text, str) or len(text.strip()) < 40:
        return []

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TOKENS_PER_DOCUMENT
    )["input_ids"]

    if len(encoded) < 20:
        return []

    best = {}

    for start in range(0, len(encoded), SEQ_LEN):
        chunk = encoded[start:start + SEQ_LEN]

        if len(chunk) < 16:
            continue

        batch = torch.tensor(
            [chunk], dtype=torch.long, device=DEVICE
        )

        with torch.inference_mode():
            out = model(
                input_ids=batch,
                output_hidden_states=True,
                use_cache=False
            )

            for layer, descriptions in targets.items():
                hidden = out.hidden_states[layer][0]

                sae = saes[layer]
                centered = hidden - sae["mean"]

                pre = torch.relu(
                    centered @ sae["W"] + sae["b"]
                )

                # Reproduce Top-K activation gating.
                _, top_indices = torch.topk(
                    pre, TOP_K, dim=-1
                )

                active_mask = torch.zeros_like(
                    pre, dtype=torch.bool
                )
                active_mask.scatter_(1, top_indices, True)

                feature_ids = [
                    d["feature"] for d in descriptions
                ]

                feature_tensor = torch.tensor(
                    feature_ids, device=DEVICE, dtype=torch.long
                )

                selected_values = pre[:, feature_tensor]
                selected_active = active_mask[:, feature_tensor]

                selected_values = selected_values * selected_active

                maxima, positions = selected_values.max(dim=0)

                for j, description in enumerate(descriptions):
                    score = float(maxima[j].item())

                    if score <= 0:
                        continue

                    local_position = int(positions[j].item())
                    global_position = start + local_position

                    key = (
                        layer,
                        description["feature"],
                        description["role"]
                    )

                    if key in best and best[key]["activation"] >= score:
                        continue

                    left = max(0, local_position - CONTEXT_WINDOW)
                    right = min(len(chunk), local_position + CONTEXT_WINDOW + 1)

                    context = tokenizer.decode(
                        chunk[left:right],
                        skip_special_tokens=True
                    ).strip()

                    active_token = tokenizer.decode(
                        [chunk[local_position]],
                        skip_special_tokens=True
                    )

                    best[key] = {
                        "corpus": corpus,
                        "document_id": str(doc_id),
                        "layer": layer,
                        "feature": description["feature"],
                        "role": description["role"],
                        "candidate_tier": description["tier"],
                        "activation": score,
                        "token_position": global_position,
                        "active_token": active_token,
                        "context": context
                    }

            del out, batch

    return list(best.values())

# --------------------------- DATASET SOURCES --------------------------------

def get_wikitext_documents(limit):
    print("\nLoading WikiText-103 TEST split...")

    ds = load_dataset(
        "wikitext",
        "wikitext-103-raw-v1",
        split="test"
    )

    docs = []
    for i, row in enumerate(ds):
        text = row.get("text", "").strip()
        if len(text) >= 100:
            docs.append((f"wikitext_test_{i}", text))
        if len(docs) >= limit:
            break

    return docs

def get_c4_documents(limit):
    print("\nStreaming C4 English VALIDATION documents...")

    ds = load_dataset(
        "allenai/c4",
        "en",
        split="validation",
        streaming=True
    )

    docs = []

    for i, row in enumerate(ds):
        text = row.get("text", "").strip()

        if len(text) < 100:
            continue

        docs.append((f"c4_validation_{i}", text))

        if len(docs) >= limit:
            break

    return docs

# --------------------------- RUN EXTRACTION ---------------------------------

all_contexts = []

def run_corpus(corpus_name, documents):
    print(f"\nExtracting activations from {corpus_name}")
    print("Documents:", len(documents))

    corpus_results = []

    for doc_id, text in tqdm(documents):
        rows = extract_document(doc_id, text, corpus_name)
        corpus_results.extend(rows)

        # Persist every document's results.
        if len(corpus_results) % 100 < max(1, len(rows)):
            pd.DataFrame(corpus_results).to_csv(
                f"{OUTPUT}/{corpus_name}_contexts_checkpoint.csv",
                index=False
            )

    result = pd.DataFrame(corpus_results)

    result.to_csv(
        f"{OUTPUT}/{corpus_name}_document_contexts.csv",
        index=False
    )

    print("Feature-document observations:", len(result))

    return result

wikitext_docs = get_wikitext_documents(MAX_DOCS_WIKITEXT)
wikitext_contexts = run_corpus("wikitext_test", wikitext_docs)

# Release dataset memory before C4.
del wikitext_docs
gc.collect()

c4_docs = get_c4_documents(MAX_DOCS_C4)
c4_contexts = run_corpus("c4_validation", c4_docs)

del c4_docs
gc.collect()

all_contexts = pd.concat(
    [wikitext_contexts, c4_contexts],
    ignore_index=True
)

all_contexts.to_csv(
    f"{OUTPUT}/all_document_contexts.csv",
    index=False
)

# --------------------------- SEMANTIC VALIDATION ----------------------------

print("\nLoading sentence embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE
)

def get_cluster_terms(texts, labels, cluster):
    selected = [
        texts[i] for i in range(len(texts))
        if labels[i] == cluster
    ]

    if len(selected) < 2:
        return ""

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=100
        )

        matrix = vectorizer.fit_transform(selected)
        terms = vectorizer.get_feature_names_out()
        weights = np.asarray(matrix.mean(axis=0)).ravel()

        top = np.argsort(weights)[::-1][:8]
        return ", ".join(terms[top])

    except Exception:
        return ""

def evaluate_group(group):
    texts = group["context"].fillna("").tolist()

    if len(texts) < MIN_DOCUMENTS:
        return None

    # Require independent documents, not repeated chunks.
    if group["document_id"].nunique() < MIN_DOCUMENTS:
        return None

    embeddings = embedder.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    n = len(texts)
    best = None

    for k in CLUSTER_COUNTS:
        if n < k * 5:
            continue

        runs = []

        for seed in SEEDS:
            km = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=10
            )

            labels = km.fit_predict(embeddings)
            sizes = np.bincount(labels, minlength=k)

            if sizes.min() < 4:
                continue

            sil = float(
                silhouette_score(
                    embeddings,
                    labels,
                    metric="cosine"
                )
            )

            runs.append({
                "labels": labels,
                "silhouette": sil,
                "sizes": sizes
            })

        if not runs:
            continue

        chosen = max(runs, key=lambda x: x["silhouette"])

        ari = [
            adjusted_rand_score(a["labels"], b["labels"])
            for i, a in enumerate(runs)
            for b in runs[i + 1:]
        ]

        seed_stability = float(np.mean(ari)) if ari else 1.0

        if best is None or chosen["silhouette"] > best["silhouette"]:
            best = {
                "k": k,
                "labels": chosen["labels"],
                "silhouette": chosen["silhouette"],
                "sizes": chosen["sizes"],
                "seed_stability": seed_stability
            }

    if best is None:
        return None

    labels = best["labels"]
    k = best["k"]

    # Bootstrap stability.
    rng = np.random.default_rng(2026)
    bootstrap_ari = []

    sample_size = max(k * 4, int(n * 0.8))

    for b in range(BOOTSTRAP_RUNS):
        idx = rng.choice(
            n, size=min(sample_size, n), replace=False
        )

        km = KMeans(
            n_clusters=k,
            random_state=1000 + b,
            n_init=10
        )

        km.fit(embeddings[idx])
        predicted = km.predict(embeddings)

        bootstrap_ari.append(
            adjusted_rand_score(labels, predicted)
        )

    bootstrap_stability = float(np.mean(bootstrap_ari))

    # Permutation test: compare observed silhouette to random
    # assignments preserving the observed cluster sizes.
    observed = best["silhouette"]
    null_scores = []

    for _ in range(PERMUTATION_RUNS):
        shuffled = rng.permutation(labels)

        try:
            null_scores.append(
                silhouette_score(
                    embeddings,
                    shuffled,
                    metric="cosine"
                )
            )
        except Exception:
            pass

    p_value = (
        1 + sum(x >= observed for x in null_scores)
    ) / (1 + len(null_scores))

    terms = {
        int(c): get_cluster_terms(texts, labels, c)
        for c in range(k)
    }

    sizes = np.bincount(labels, minlength=k)

    cluster_contexts = group.copy()
    cluster_contexts["semantic_cluster"] = labels

    return {
        "k": int(k),
        "n_contexts": int(n),
        "n_documents": int(group["document_id"].nunique()),
        "silhouette": observed,
        "permutation_p": float(p_value),
        "seed_stability": seed_stability,
        "bootstrap_stability": bootstrap_stability,
        "min_cluster_size": int(sizes.min()),
        "cluster_sizes": json.dumps(sizes.tolist()),
        "cluster_terms": json.dumps(terms),
        "cluster_contexts": cluster_contexts
    }

# --------------------------- EVALUATE ALL FEATURES ---------------------------

metric_records = []
clustered_contexts = []

group_columns = [
    "corpus", "layer", "feature", "role"
]

groups = all_contexts.groupby(group_columns)

print("\nEvaluating candidates and matched controls...")

for (corpus, layer, feature, role), group in tqdm(
    groups, total=groups.ngroups
):
    result = evaluate_group(group)

    if result is None:
        metric_records.append({
            "corpus": corpus,
            "layer": layer,
            "feature": feature,
            "role": role,
            "n_contexts": len(group),
            "n_documents": group["document_id"].nunique(),
            "evaluable": False
        })
        continue

    contexts_with_clusters = result.pop("cluster_contexts")

    metric_records.append({
        "corpus": corpus,
        "layer": layer,
        "feature": feature,
        "role": role,
        "evaluable": True,
        **result
    })

    clustered_contexts.append(contexts_with_clusters)

metrics = pd.DataFrame(metric_records)

metrics.to_csv(
    f"{OUTPUT}/validation_metrics.csv",
    index=False
)

if clustered_contexts:
    pd.concat(
        clustered_contexts,
        ignore_index=True
    ).to_csv(
        f"{OUTPUT}/validated_context_clusters.csv",
        index=False
    )

# --------------------------- CANDIDATE VS CONTROL ----------------------------

evaluable = metrics[
    metrics["evaluable"] == True
].copy()

candidate_results = evaluable[
    evaluable["role"] == "candidate"
].copy()

control_results = evaluable[
    evaluable["role"] == "matched_control"
].copy()

# Strong replication criteria are deliberately conservative.
candidate_results["passes_semantic_test"] = (
    (candidate_results["n_documents"] >= MIN_DOCUMENTS) &
    (candidate_results["k"] >= 2) &
    (candidate_results["silhouette"] >= 0.20) &
    (candidate_results["permutation_p"] <= 0.05) &
    (candidate_results["bootstrap_stability"] >= 0.60) &
    (candidate_results["seed_stability"] >= 0.50) &
    (candidate_results["min_cluster_size"] >= 4)
)

# Cross-corpus replication requires passing in BOTH corpora.
replication = candidate_results.pivot_table(
    index=["layer", "feature"],
    columns="corpus",
    values="passes_semantic_test",
    aggfunc="first"
).reset_index()

for corpus in ["wikitext_test", "c4_validation"]:
    if corpus not in replication.columns:
        replication[corpus] = False

replication["replicated_across_corpora"] = (
    replication["wikitext_test"].fillna(False) &
    replication["c4_validation"].fillna(False)
)

replication.to_csv(
    f"{OUTPUT}/cross_corpus_replication.csv",
    index=False
)

# Compare candidate and control pass rates.
candidate_pass_rate = (
    candidate_results["passes_semantic_test"].mean()
    if len(candidate_results) else np.nan
)

# Controls use the same criteria.
control_results["passes_semantic_test"] = (
    (control_results["n_documents"] >= MIN_DOCUMENTS) &
    (control_results["k"] >= 2) &
    (control_results["silhouette"] >= 0.20) &
    (control_results["permutation_p"] <= 0.05) &
    (control_results["bootstrap_stability"] >= 0.60) &
    (control_results["seed_stability"] >= 0.50) &
    (control_results["min_cluster_size"] >= 4)
)

control_pass_rate = (
    control_results["passes_semantic_test"].mean()
    if len(control_results) else np.nan
)

# --------------------------- FINAL EVIDENCE TABLE ----------------------------

summary = candidate_results.merge(
    replication,
    on=["layer", "feature"],
    how="left"
)

summary["evidence_class"] = "not_replicated"

summary.loc[
    summary["passes_semantic_test"] == True,
    "evidence_class"
] = "single_corpus_candidate"

summary.loc[
    summary["replicated_across_corpora"] == True,
    "evidence_class"
] = "cross_corpus_replicated_candidate"

summary = summary.sort_values(
    [
        "replicated_across_corpora",
        "bootstrap_stability",
        "silhouette"
    ],
    ascending=[False, False, False]
)

summary.to_csv(
    f"{OUTPUT}/candidate_evidence_ranking.csv",
    index=False
)

# --------------------------- SUMMARY REPORT ---------------------------------

report = {
    "experiment": "Experiment 03 - Rigorous Polysemanticity Validation",
    "model": MODEL_ID,
    "revision": REVISION,
    "sae_checkpoints_reused": 36,
    "candidate_count": int(len(candidate_df)),
    "matched_control_count": int(len(controls_df)),
    "wikitext_test_documents": int(len(wikitext_contexts["document_id"].unique())),
    "c4_validation_documents": int(len(c4_contexts["document_id"].unique())),
    "candidate_evaluations": int(len(candidate_results)),
    "control_evaluations": int(len(control_results)),
    "candidate_pass_rate": (
        float(candidate_pass_rate)
        if not np.isnan(candidate_pass_rate) else None
    ),
    "control_pass_rate": (
        float(control_pass_rate)
        if not np.isnan(control_pass_rate) else None
    ),
    "cross_corpus_replicated_candidates": int(
        replication["replicated_across_corpora"].sum()
    ),
    "permutation_runs": PERMUTATION_RUNS,
    "bootstrap_runs": BOOTSTRAP_RUNS,
    "important_limitation": (
        "Semantic replication and clustering do not establish causal "
        "polysemanticity. Causal activation interventions are a separate "
        "required validation stage."
    )
}

with open(f"{OUTPUT}/experiment_03_summary.json", "w") as f:
    json.dump(report, f, indent=2)

# --------------------------- SAVE CONFIGURATION ------------------------------

with open(f"{OUTPUT}/experiment_03_configuration.json", "w") as f:
    json.dump({
        "model": MODEL_ID,
        "revision": REVISION,
        "candidate_file": CANDIDATE_FILE,
        "feature_file": FEATURE_FILE,
        "wikitext_docs": MAX_DOCS_WIKITEXT,
        "c4_docs": MAX_DOCS_C4,
        "sequence_length": SEQ_LEN,
        "context_window": CONTEXT_WINDOW,
        "cluster_counts": CLUSTER_COUNTS,
        "seeds": SEEDS,
        "permutation_runs": PERMUTATION_RUNS,
        "bootstrap_runs": BOOTSTRAP_RUNS,
        "minimum_documents": MIN_DOCUMENTS
    }, f, indent=2)

# --------------------------- DISPLAY -----------------------------------------

print("\n" + "=" * 75)
print("EXPERIMENT 03 COMPLETE")
print("=" * 75)

print(json.dumps(report, indent=2))

print("\nCROSS-CORPUS REPLICATED CANDIDATES")

display(
    summary[
        summary["replicated_across_corpora"] == True
    ][
        [
            "layer", "feature", "corpus", "n_documents",
            "silhouette", "permutation_p",
            "bootstrap_stability", "evidence_class"
        ]
    ].head(30)
)

print("\nTOP CANDIDATE EVIDENCE")

display(
    summary[
        [
            "layer", "feature", "corpus",
            "n_documents", "k", "silhouette",
            "permutation_p", "seed_stability",
            "bootstrap_stability", "evidence_class"
        ]
    ].head(30)
)

print("\nCandidate pass rate:", candidate_pass_rate)
print("Matched-control pass rate:", control_pass_rate)

print("\nAll outputs saved to:")
print(OUTPUT)

# --------------------------- ZIP RESULTS -------------------------------------

import shutil

zip_base = f"{V2}/Experiment_03_Validation_{RUN_ID}"

shutil.make_archive(
    zip_base,
    "zip",
    OUTPUT
)

print("\nZIP:", zip_base + ".zip")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("Automatic download unavailable:", e)
    print("ZIP remains saved in Google Drive.")


In [ ]:

# FIX: Use the fully qualified WikiText repository.
# Reuse the model and SAEs already loaded in memory.

def get_wikitext_documents(limit):
    print("Loading WikiText-103 TEST split...")

    ds = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="test"
    )

    docs = []

    for i, row in enumerate(ds):
        text = row.get("text", "").strip()

        if len(text) >= 100:
            docs.append((f"wikitext_test_{i}", text))

        if len(docs) >= limit:
            break

    print("WikiText documents:", len(docs))
    return docs


# Resume from the failed dataset-loading stage.
wikitext_docs = get_wikitext_documents(MAX_DOCS_WIKITEXT)

wikitext_contexts = run_corpus(
    "wikitext_test",
    wikitext_docs
)

del wikitext_docs
gc.collect()

# Continue with independent C4 validation.
c4_docs = get_c4_documents(MAX_DOCS_C4)

c4_contexts = run_corpus(
    "c4_validation",
    c4_docs
)

del c4_docs
gc.collect()

# Combine and persist both corpora.
all_contexts = pd.concat(
    [wikitext_contexts, c4_contexts],
    ignore_index=True
)

all_contexts.to_csv(
    f"{OUTPUT}/all_document_contexts.csv",
    index=False
)

print("Both corpus extraction stages completed.")
print("Total observations:", len(all_contexts))


In [ ]:
# --------------------------- SEMANTIC VALIDATION ----------------------------

print("\nLoading sentence embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE
)

def get_cluster_terms(texts, labels, cluster):
    selected = [
        texts[i] for i in range(len(texts))
        if labels[i] == cluster
    ]

    if len(selected) < 2:
        return ""

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=100
        )

        matrix = vectorizer.fit_transform(selected)
        terms = vectorizer.get_feature_names_out()
        weights = np.asarray(matrix.mean(axis=0)).ravel()

        top = np.argsort(weights)[::-1][:8]
        return ", ".join(terms[top])

    except Exception:
        return ""

def evaluate_group(group):
    texts = group["context"].fillna("").tolist()

    if len(texts) < MIN_DOCUMENTS:
        return None

    # Require independent documents, not repeated chunks.
    if group["document_id"].nunique() < MIN_DOCUMENTS:
        return None

    embeddings = embedder.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    n = len(texts)
    best = None

    for k in CLUSTER_COUNTS:
        if n < k * 5:
            continue

        runs = []

        for seed in SEEDS:
            km = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=10
            )

            labels = km.fit_predict(embeddings)
            sizes = np.bincount(labels, minlength=k)

            if sizes.min() < 4:
                continue

            sil = float(
                silhouette_score(
                    embeddings,
                    labels,
                    metric="cosine"
                )
            )

            runs.append({
                "labels": labels,
                "silhouette": sil,
                "sizes": sizes
            })

        if not runs:
            continue

        chosen = max(runs, key=lambda x: x["silhouette"])

        ari = [
            adjusted_rand_score(a["labels"], b["labels"])
            for i, a in enumerate(runs)
            for b in runs[i + 1:]
        ]

        seed_stability = float(np.mean(ari)) if ari else 1.0

        if best is None or chosen["silhouette"] > best["silhouette"]:
            best = {
                "k": k,
                "labels": chosen["labels"],
                "silhouette": chosen["silhouette"],
                "sizes": chosen["sizes"],
                "seed_stability": seed_stability
            }

    if best is None:
        return None

    labels = best["labels"]
    k = best["k"]

    # Bootstrap stability.
    rng = np.random.default_rng(2026)
    bootstrap_ari = []

    sample_size = max(k * 4, int(n * 0.8))

    for b in range(BOOTSTRAP_RUNS):
        idx = rng.choice(
            n, size=min(sample_size, n), replace=False
        )

        km = KMeans(
            n_clusters=k,
            random_state=1000 + b,
            n_init=10
        )

        km.fit(embeddings[idx])
        predicted = km.predict(embeddings)

        bootstrap_ari.append(
            adjusted_rand_score(labels, predicted)
        )

    bootstrap_stability = float(np.mean(bootstrap_ari))

    # Permutation test: compare observed silhouette to random
    # assignments preserving the observed cluster sizes.
    observed = best["silhouette"]
    null_scores = []

    for _ in range(PERMUTATION_RUNS):
        shuffled = rng.permutation(labels)

        try:
            null_scores.append(
                silhouette_score(
                    embeddings,
                    shuffled,
                    metric="cosine"
                )
            )
        except Exception:
            pass

    p_value = (
        1 + sum(x >= observed for x in null_scores)
    ) / (1 + len(null_scores))

    terms = {
        int(c): get_cluster_terms(texts, labels, c)
        for c in range(k)
    }

    sizes = np.bincount(labels, minlength=k)

    cluster_contexts = group.copy()
    cluster_contexts["semantic_cluster"] = labels

    return {
        "k": int(k),
        "n_contexts": int(n),
        "n_documents": int(group["document_id"].nunique()),
        "silhouette": observed,
        "permutation_p": float(p_value),
        "seed_stability": seed_stability,
        "bootstrap_stability": bootstrap_stability,
        "min_cluster_size": int(sizes.min()),
        "cluster_sizes": json.dumps(sizes.tolist()),
        "cluster_terms": json.dumps(terms),
        "cluster_contexts": cluster_contexts
    }

# --------------------------- EVALUATE ALL FEATURES ---------------------------

metric_records = []
clustered_contexts = []

group_columns = [
    "corpus", "layer", "feature", "role"
]

groups = all_contexts.groupby(group_columns)

print("\nEvaluating candidates and matched controls...")

for (corpus, layer, feature, role), group in tqdm(
    groups, total=groups.ngroups
):
    result = evaluate_group(group)

    if result is None:
        metric_records.append({
            "corpus": corpus,
            "layer": layer,
            "feature": feature,
            "role": role,
            "n_contexts": len(group),
            "n_documents": group["document_id"].nunique(),
            "evaluable": False
        })
        continue

    contexts_with_clusters = result.pop("cluster_contexts")

    metric_records.append({
        "corpus": corpus,
        "layer": layer,
        "feature": feature,
        "role": role,
        "evaluable": True,
        **result
    })

    clustered_contexts.append(contexts_with_clusters)

metrics = pd.DataFrame(metric_records)

metrics.to_csv(
    f"{OUTPUT}/validation_metrics.csv",
    index=False
)

if clustered_contexts:
    pd.concat(
        clustered_contexts,
        ignore_index=True
    ).to_csv(
        f"{OUTPUT}/validated_context_clusters.csv",
        index=False
    )

# --------------------------- CANDIDATE VS CONTROL ----------------------------

evaluable = metrics[
    metrics["evaluable"] == True
].copy()

candidate_results = evaluable[
    evaluable["role"] == "candidate"
].copy()

control_results = evaluable[
    evaluable["role"] == "matched_control"
].copy()

# Strong replication criteria are deliberately conservative.
candidate_results["passes_semantic_test"] = (
    (candidate_results["n_documents"] >= MIN_DOCUMENTS) &
    (candidate_results["k"] >= 2) &
    (candidate_results["silhouette"] >= 0.20) &
    (candidate_results["permutation_p"] <= 0.05) &
    (candidate_results["bootstrap_stability"] >= 0.60) &
    (candidate_results["seed_stability"] >= 0.50) &
    (candidate_results["min_cluster_size"] >= 4)
)

# Cross-corpus replication requires passing in BOTH corpora.
replication = candidate_results.pivot_table(
    index=["layer", "feature"],
    columns="corpus",
    values="passes_semantic_test",
    aggfunc="first"
).reset_index()

for corpus in ["wikitext_test", "c4_validation"]:
    if corpus not in replication.columns:
        replication[corpus] = False

replication["replicated_across_corpora"] = (
    replication["wikitext_test"].fillna(False) &
    replication["c4_validation"].fillna(False)
)

replication.to_csv(
    f"{OUTPUT}/cross_corpus_replication.csv",
    index=False
)

# Compare candidate and control pass rates.
candidate_pass_rate = (
    candidate_results["passes_semantic_test"].mean()
    if len(candidate_results) else np.nan
)

# Controls use the same criteria.
control_results["passes_semantic_test"] = (
    (control_results["n_documents"] >= MIN_DOCUMENTS) &
    (control_results["k"] >= 2) &
    (control_results["silhouette"] >= 0.20) &
    (control_results["permutation_p"] <= 0.05) &
    (control_results["bootstrap_stability"] >= 0.60) &
    (control_results["seed_stability"] >= 0.50) &
    (control_results["min_cluster_size"] >= 4)
)

control_pass_rate = (
    control_results["passes_semantic_test"].mean()
    if len(control_results) else np.nan
)

# --------------------------- FINAL EVIDENCE TABLE ----------------------------

summary = candidate_results.merge(
    replication,
    on=["layer", "feature"],
    how="left"
)

summary["evidence_class"] = "not_replicated"

summary.loc[
    summary["passes_semantic_test"] == True,
    "evidence_class"
] = "single_corpus_candidate"

summary.loc[
    summary["replicated_across_corpora"] == True,
    "evidence_class"
] = "cross_corpus_replicated_candidate"

summary = summary.sort_values(
    [
        "replicated_across_corpora",
        "bootstrap_stability",
        "silhouette"
    ],
    ascending=[False, False, False]
)

summary.to_csv(
    f"{OUTPUT}/candidate_evidence_ranking.csv",
    index=False
)

# --------------------------- SUMMARY REPORT ---------------------------------

report = {
    "experiment": "Experiment 03 - Rigorous Polysemanticity Validation",
    "model": MODEL_ID,
    "revision": REVISION,
    "sae_checkpoints_reused": 36,
    "candidate_count": int(len(candidate_df)),
    "matched_control_count": int(len(controls_df)),
    "wikitext_test_documents": int(len(wikitext_contexts["document_id"].unique())),
    "c4_validation_documents": int(len(c4_contexts["document_id"].unique())),
    "candidate_evaluations": int(len(candidate_results)),
    "control_evaluations": int(len(control_results)),
    "candidate_pass_rate": (
        float(candidate_pass_rate)
        if not np.isnan(candidate_pass_rate) else None
    ),
    "control_pass_rate": (
        float(control_pass_rate)
        if not np.isnan(control_pass_rate) else None
    ),
    "cross_corpus_replicated_candidates": int(
        replication["replicated_across_corpora"].sum()
    ),
    "permutation_runs": PERMUTATION_RUNS,
    "bootstrap_runs": BOOTSTRAP_RUNS,
    "important_limitation": (
        "Semantic replication and clustering do not establish causal "
        "polysemanticity. Causal activation interventions are a separate "
        "required validation stage."
    )
}

with open(f"{OUTPUT}/experiment_03_summary.json", "w") as f:
    json.dump(report, f, indent=2)

# --------------------------- SAVE CONFIGURATION ------------------------------

with open(f"{OUTPUT}/experiment_03_configuration.json", "w") as f:
    json.dump({
        "model": MODEL_ID,
        "revision": REVISION,
        "candidate_file": CANDIDATE_FILE,
        "feature_file": FEATURE_FILE,
        "wikitext_docs": MAX_DOCS_WIKITEXT,
        "c4_docs": MAX_DOCS_C4,
        "sequence_length": SEQ_LEN,
        "context_window": CONTEXT_WINDOW,
        "cluster_counts": CLUSTER_COUNTS,
        "seeds": SEEDS,
        "permutation_runs": PERMUTATION_RUNS,
        "bootstrap_runs": BOOTSTRAP_RUNS,
        "minimum_documents": MIN_DOCUMENTS
    }, f, indent=2)

# --------------------------- DISPLAY -----------------------------------------

print("\n" + "=" * 75)
print("EXPERIMENT 03 COMPLETE")
print("=" * 75)

print(json.dumps(report, indent=2))

print("\nCROSS-CORPUS REPLICATED CANDIDATES")

display(
    summary[
        summary["replicated_across_corpora"] == True
    ][
        [
            "layer", "feature", "corpus", "n_documents",
            "silhouette", "permutation_p",
            "bootstrap_stability", "evidence_class"
        ]
    ].head(30)
)

print("\nTOP CANDIDATE EVIDENCE")

display(
    summary[
        [
            "layer", "feature", "corpus",
            "n_documents", "k", "silhouette",
            "permutation_p", "seed_stability",
            "bootstrap_stability", "evidence_class"
        ]
    ].head(30)
)

print("\nCandidate pass rate:", candidate_pass_rate)
print("Matched-control pass rate:", control_pass_rate)

print("\nAll outputs saved to:")
print(OUTPUT)

# --------------------------- ZIP RESULTS -------------------------------------

import shutil

zip_base = f"{V2}/Experiment_03_Validation_{RUN_ID}"

shutil.make_archive(
    zip_base,
    "zip",
    OUTPUT
)

print("\nZIP:", zip_base + ".zip")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("Automatic download unavailable:", e)
    print("ZIP remains saved in Google Drive.")


In [ ]:

# ================================================================
# EXPERIMENT 03B
# Corrected, Cross-Corpus and Causal Validation of SAE Polysemanticity
#
# Model: Qwen/Qwen2.5-3B
# Revision: 3aab1f1954e9cc14eb9509a215f9e5ca08227a9b
# SAE: 2048 -> 8192, Top-K = 32
# Existing SAE checkpoints: 36
#
# No SAE retraining.
# ================================================================

import os, sys, re, gc, json, glob, time, math, random, hashlib, zipfile, warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ----------------------- CONFIGURATION ----------------------------

SEED = 2026
random.seed(SEED)

ROOT = "/content/drive/MyDrive/Mapping_Feature_Drift"
PROJECT_DIR = f"{ROOT}/sae_research_100k_v2"

SAE_DIR = f"{PROJECT_DIR}/saes"
DEEP_DIR = f"{PROJECT_DIR}/experiment_02_polysemanticity/deep_same_corpus"

MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

D_MODEL = 2048
D_SAE = 8192
TOP_K = 32

N_WIKITEXT_ARTICLES = 500
N_C4_DOCUMENTS = 500

MAX_SEQ_LEN = 256
MAX_WINDOWS_PER_DOCUMENT = 2
MAX_CONTEXTS_PER_FEATURE_DOCUMENT = 3

MIN_CONTEXTS = 20
MAX_CLUSTER_K = 4
KMEANS_SEEDS = [11, 29, 47, 83, 101]
BOOTSTRAP_RUNS = 20
PERMUTATION_RUNS = 1999

# Causal intervention is restricted to the strongest candidates.
CAUSAL_FEATURE_LIMIT = 10
CAUSAL_CONTEXTS_PER_FEATURE = 3

OUTPUT_ROOT = f"{PROJECT_DIR}/experiment_03B_corrected"

# ----------------------- INSTALL PACKAGES -------------------------

import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "accelerate",
    "sentence-transformers", "scikit-learn",
    "pandas", "numpy", "scipy", "tqdm"
])

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import TfidfVectorizer
from statsmodels.stats.multitest import multipletests

# ----------------------- DRIVE MOUNT ------------------------------

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

assert os.path.isdir(PROJECT_DIR), f"Project directory missing: {PROJECT_DIR}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUT = f"{OUTPUT_ROOT}/run_{RUN_ID}"
os.makedirs(OUT, exist_ok=True)

print("\nOutput:", OUT)

# ----------------------- GPU --------------------------------------

assert torch.cuda.is_available(), "A CUDA GPU is required."

DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

# ----------------------- LOCATE INPUT FILES -----------------------

def find_csv_with_columns(directory, required, preferred_names=None):
    preferred_names = preferred_names or []
    files = glob.glob(directory + "/**/*.csv", recursive=True)

    for name in preferred_names:
        for path in files:
            if os.path.basename(path).lower() == name.lower():
                try:
                    cols = set(pd.read_csv(path, nrows=5).columns)
                    if set(required).issubset(cols):
                        return path
                except Exception:
                    pass

    for path in files:
        try:
            cols = set(pd.read_csv(path, nrows=5).columns)
            if set(required).issubset(cols):
                return path
        except Exception:
            pass

    return None


candidate_path = find_csv_with_columns(
    DEEP_DIR,
    ["layer", "feature"],
    [
        "deep_polysemanticity_metrics.csv",
        "candidate_features.csv",
        "deep_candidates.csv"
    ]
)

selected_path = find_csv_with_columns(
    PROJECT_DIR,
    ["layer", "feature"],
    ["features_selected.csv"]
)

if candidate_path is None:
    raise FileNotFoundError(
        "Could not locate the Experiment 02B candidate CSV. "
        "Expected a CSV containing layer and feature columns."
    )

if selected_path is None:
    raise FileNotFoundError(
        "Could not locate features_selected.csv. "
        "This file is required to construct matched controls."
    )

print("\nCandidate source:", candidate_path)
print("Selected features:", selected_path)

candidate_raw = pd.read_csv(candidate_path)
selected = pd.read_csv(selected_path)

# ----------------------- NORMALIZE CANDIDATES ---------------------

def normalize_feature_table(df):
    df = df.copy()
    df["layer"] = pd.to_numeric(df["layer"], errors="coerce")
    df["feature"] = pd.to_numeric(df["feature"], errors="coerce")
    df = df.dropna(subset=["layer", "feature"])
    df["layer"] = df["layer"].astype(int)
    df["feature"] = df["feature"].astype(int)
    return df

candidate_raw = normalize_feature_table(candidate_raw)
selected = normalize_feature_table(selected)

# Identify candidate tier/status column.
tier_col = next(
    (c for c in ["candidate_tier", "tier", "candidate_status", "status"]
     if c in candidate_raw.columns),
    None
)

if tier_col:
    tiers = candidate_raw[tier_col].astype(str).str.lower()
    candidate_df = candidate_raw[
        tiers.str.contains("strong|provisional|candidate", regex=True)
    ].copy()
else:
    flag_col = next(
        (c for c in ["is_candidate", "candidate", "selected_candidate"]
         if c in candidate_raw.columns),
        None
    )
    if flag_col:
        candidate_df = candidate_raw[
            candidate_raw[flag_col].astype(str).str.lower().isin(
                ["true", "1", "yes"]
            )
        ].copy()
    else:
        raise ValueError(
            "Candidate CSV has no recognizable candidate-tier or candidate "
            "flag column. Inspect candidate CSV columns before proceeding."
        )

candidate_df = candidate_df.drop_duplicates(["layer", "feature"])

if len(candidate_df) != 79:
    print(
        f"WARNING: Expected 79 candidates from Experiment 02B; "
        f"found {len(candidate_df)}."
    )

# Recover prevalence from the selected-feature table.
prevalence_col = next(
    (c for c in [
        "sequence_prevalence",
        "document_prevalence",
        "activation_prevalence",
        "prevalence"
    ] if c in selected.columns),
    None
)

if prevalence_col is None:
    raise ValueError(
        "No prevalence column found in features_selected.csv. "
        "A prevalence-matched control set cannot be constructed reliably."
    )

selected[prevalence_col] = pd.to_numeric(
    selected[prevalence_col], errors="coerce"
)

candidate_df = candidate_df.merge(
    selected[["layer", "feature", prevalence_col]].drop_duplicates(
        ["layer", "feature"]
    ),
    on=["layer", "feature"],
    how="left",
    validate="one_to_one"
)

candidate_df = candidate_df.dropna(subset=[prevalence_col]).copy()

# ----------------------- MATCHED CONTROLS -------------------------

# One control per candidate, from the same SAE layer, with similar
# activation prevalence. Controls cannot be candidate features.

candidate_keys = set(
    zip(candidate_df["layer"], candidate_df["feature"])
)

control_pool = selected.dropna(subset=[prevalence_col]).copy()
control_pool = control_pool[
    ~control_pool.apply(
        lambda r: (int(r.layer), int(r.feature)) in candidate_keys,
        axis=1
    )
]

controls = []
used_controls = set()

for _, cand in candidate_df.sort_values(
    prevalence_col
).iterrows():

    layer = int(cand.layer)
    prevalence = float(cand[prevalence_col])

    pool = control_pool[
        (control_pool.layer == layer) &
        (~control_pool.apply(
            lambda r: (int(r.layer), int(r.feature)) in used_controls,
            axis=1
        ))
    ].copy()

    if pool.empty:
        continue

    pool["match_distance"] = (
        pool[prevalence_col].astype(float) - prevalence
    ).abs()

    chosen = pool.sort_values("match_distance").iloc[0]

    controls.append({
        "candidate_layer": layer,
        "candidate_feature": int(cand.feature),
        "layer": layer,
        "feature": int(chosen.feature),
        "sequence_prevalence": float(chosen[prevalence_col]),
        "candidate_prevalence": prevalence
    })

    used_controls.add((layer, int(chosen.feature)))

control_df = pd.DataFrame(controls)

if len(control_df) != len(candidate_df):
    print(
        "WARNING: Some candidates could not receive unique matched controls."
    )

candidate_df["role"] = "candidate"
candidate_df["candidate_layer"] = candidate_df["layer"]
candidate_df["candidate_feature"] = candidate_df["feature"]

control_df["role"] = "matched_control"

print("\nCandidates:", len(candidate_df))
print("Matched controls:", len(control_df))

# Persist matching decisions.
candidate_df.to_csv(f"{OUT}/candidate_features.csv", index=False)
control_df.to_csv(f"{OUT}/matched_controls.csv", index=False)

# ----------------------- LOAD SAE CHECKPOINTS ---------------------

def get_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model_state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
    return checkpoint


def find_tensor(state, aliases):
    for alias in aliases:
        if alias in state and torch.is_tensor(state[alias]):
            return state[alias]

    for key, value in state.items():
        if torch.is_tensor(value):
            for alias in aliases:
                if key.endswith(alias):
                    return value

    return None


def load_sae(layer):
    path = f"{SAE_DIR}/sae_layer_{layer:02d}.pt"

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing SAE checkpoint: {path}")

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )
    state = get_state_dict(checkpoint)

    W_enc = find_tensor(
        state, ["W_enc", "encoder.weight", "encoder.weight_orig"]
    )
    b_enc = find_tensor(
        state, ["b_enc", "encoder.bias"]
    )
    W_dec = find_tensor(
        state, ["W_dec", "decoder.weight", "decoder.weight_orig"]
    )
    b_dec = find_tensor(
        state, ["b_dec", "decoder.bias"]
    )
    mean = find_tensor(
        state, [
            "mean", "activation_mean", "input_mean",
            "running_mean", "preprocessing_mean"
        ]
    )

    if W_enc is None or b_enc is None or W_dec is None:
        raise ValueError(
            f"Could not identify SAE tensors in {path}. "
            f"Available keys: {list(state.keys())[:30]}"
        )

    # Encoder canonical shape: [d_sae, d_model].
    if W_enc.shape == (D_MODEL, D_SAE):
        W_enc = W_enc.T

    # Decoder canonical shape: [d_sae, d_model].
    if W_dec.shape == (D_MODEL, D_SAE):
        W_dec = W_dec.T

    if W_enc.shape != (D_SAE, D_MODEL):
        raise ValueError(f"Unexpected encoder shape: {W_enc.shape}")

    if W_dec.shape != (D_SAE, D_MODEL):
        raise ValueError(f"Unexpected decoder shape: {W_dec.shape}")

    if mean is None:
        mean = torch.zeros(D_MODEL)

    return {
        "W_enc": W_enc.to(DEVICE, dtype=torch.float16).contiguous(),
        "b_enc": b_enc.to(DEVICE, dtype=torch.float16).contiguous(),
        "W_dec": W_dec.to(DEVICE, dtype=torch.float16).contiguous(),
        "b_dec": (
            b_dec.to(DEVICE, dtype=torch.float16)
            if b_dec is not None else None
        ),
        "mean": mean.to(DEVICE, dtype=torch.float16).reshape(-1)
    }


needed_layers = sorted(
    set(candidate_df.layer.astype(int)) |
    set(control_df.layer.astype(int))
)

assert all(1 <= x <= 36 for x in needed_layers)

saes = {}

for layer in tqdm(needed_layers, desc="Loading SAE checkpoints"):
    saes[layer] = load_sae(layer)

print("Loaded SAE layers:", len(saes))

# ----------------------- LOAD MODEL -------------------------------

print("\nLoading Qwen2.5-3B...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded.")

# ----------------------- LOAD CORPORA -----------------------------

def make_wikitext_articles():
    print("\nLoading WikiText-103 test split...")

    ds = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="test"
    )

    articles = []
    title = None
    lines = []

    # Only a single-equals heading starts a new article.
    heading_re = re.compile(r"^\s*=\s*([^=].*?)\s*=\s*$")

    def flush():
        nonlocal title, lines
        body = "\n".join(lines).strip()

        if len(body) >= 300:
            articles.append({
                "document_id": f"wiki_{len(articles)}",
                "title": title or "Untitled",
                "text": body
            })

        lines = []

    for row in ds:
        line = str(row.get("text", "")).strip()

        match = heading_re.match(line)

        if match:
            flush()
            title = match.group(1).strip()
        elif line:
            lines.append(line)

    flush()

    # Deduplicate articles.
    unique = {}
    for article in articles:
        key = hashlib.sha256(
            article["text"].lower().encode("utf-8")
        ).hexdigest()
        unique[key] = article

    articles = list(unique.values())
    random.Random(SEED).shuffle(articles)

    return articles[:N_WIKITEXT_ARTICLES]


def make_c4_documents():
    print("\nLoading shuffled C4 validation documents...")

    ds = load_dataset(
        "allenai/c4",
        "en",
        split="validation",
        streaming=True
    )

    ds = ds.shuffle(seed=SEED, buffer_size=10000)

    documents = []
    seen_hashes = set()
    seen_urls = set()

    for row in tqdm(ds, desc="Sampling C4"):
        text = str(row.get("text", "")).strip()
        url = str(row.get("url", "")).strip()

        if len(text) < 300:
            continue

        if url and url in seen_urls:
            continue

        digest = hashlib.sha256(
            re.sub(r"\s+", " ", text.lower()).encode("utf-8")
        ).hexdigest()

        if digest in seen_hashes:
            continue

        seen_hashes.add(digest)

        if url:
            seen_urls.add(url)

        documents.append({
            "document_id": f"c4_{len(documents)}",
            "title": url,
            "text": text
        })

        if len(documents) >= N_C4_DOCUMENTS:
            break

    return documents


corpora = {
    "wikitext_test": make_wikitext_articles(),
    "c4_validation": make_c4_documents()
}

for name, docs in corpora.items():
    print(name, "documents:", len(docs))

    if len(docs) < 100:
        raise RuntimeError(
            f"Only {len(docs)} documents collected for {name}."
        )

# ----------------------- FEATURE INDEX ----------------------------

feature_rows = []

for _, r in candidate_df.iterrows():
    feature_rows.append({
        "layer": int(r.layer),
        "feature": int(r.feature),
        "role": "candidate",
        "candidate_layer": int(r.layer),
        "candidate_feature": int(r.feature),
        "candidate_tier": str(r.get(tier_col, "unknown"))
        if tier_col else "unknown"
    })

for _, r in control_df.iterrows():
    feature_rows.append({
        "layer": int(r.layer),
        "feature": int(r.feature),
        "role": "matched_control",
        "candidate_layer": int(r.candidate_layer),
        "candidate_feature": int(r.candidate_feature),
        "candidate_tier": "matched_control"
    })

feature_table = pd.DataFrame(feature_rows).drop_duplicates(
    ["layer", "feature", "role"]
)

# ----------------------- TOKEN WINDOWS ----------------------------

def make_windows(text):
    ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    if len(ids) < 30:
        return []

    starts = [0]

    if len(ids) > MAX_SEQ_LEN:
        middle = max(0, (len(ids) - MAX_SEQ_LEN) // 2)
        if middle not in starts:
            starts.append(middle)

    starts = starts[:MAX_WINDOWS_PER_DOCUMENT]

    windows = []

    for start in starts:
        chunk = ids[start:start + MAX_SEQ_LEN]

        if len(chunk) < 30:
            continue

        windows.append({
            "input_ids": chunk,
            "window_start": start,
            "window_text": tokenizer.decode(
                chunk,
                skip_special_tokens=True
            )
        })

    return windows


def get_context(ids, position, radius=35):
    left = max(0, position - radius)
    right = min(len(ids), position + radius + 1)

    return tokenizer.decode(
        ids[left:right],
        skip_special_tokens=True
    )


# ----------------------- EXTRACT FEATURE ACTIVATIONS ---------------

# The SAE layer numbering follows the checkpoint convention:
# SAE layer 1 is applied to the output of transformer block 1.
# In Hugging Face hidden_states, index 0 is the embedding output,
# so hidden_states[layer] is the output of block `layer`.
#
# Do not add another +1 to this 1-based layer number.

features_by_layer = defaultdict(list)

for _, row in feature_table.iterrows():
    features_by_layer[int(row.layer)].append(row.to_dict())


all_contexts = []

for corpus_name, documents in corpora.items():

    print(f"\nExtracting {corpus_name}...")

    for doc in tqdm(documents, desc=corpus_name):

        windows = make_windows(doc["text"])

        for window_idx, window in enumerate(windows):

            ids = window["input_ids"]

            input_ids = torch.tensor(
                [ids], device=DEVICE, dtype=torch.long
            )

            with torch.inference_mode():
                outputs = model(
                    input_ids=input_ids,
                    output_hidden_states=True,
                    use_cache=False
                )

            for layer, layer_features in features_by_layer.items():

                # Correct post-block residual stream for this SAE.
                hidden = outputs.hidden_states[layer][0]

                hidden = hidden.to(torch.float16)

                sae = saes[layer]

                centered = hidden - sae["mean"]

                pre = F.linear(
                    centered,
                    sae["W_enc"],
                    sae["b_enc"]
                )

                pre = F.relu(pre)

                values, indices = torch.topk(
                    pre, k=TOP_K, dim=-1
                )

                indices_cpu = indices.cpu().numpy()
                values_cpu = values.float().cpu().numpy()

                for feature_info in layer_features:

                    feature = int(feature_info["feature"])

                    positions = np.where(
                        indices_cpu == feature
                    )

                    token_positions = positions[0]

                    if len(token_positions) == 0:
                        continue

                    acts = values_cpu[
                        token_positions,
                        positions[1]
                    ]

                    # Keep the strongest activations per feature/document/window.
                    order = np.argsort(acts)[::-1][
                        :MAX_CONTEXTS_PER_FEATURE_DOCUMENT
                    ]

                    for ix in order:
                        pos = int(token_positions[ix])
                        activation = float(acts[ix])

                        context = get_context(ids, pos)

                        if len(context.strip()) < 20:
                            continue

                        all_contexts.append({
                            "corpus": corpus_name,
                            "document_id": doc["document_id"],
                            "document_title": doc.get("title", ""),
                            "layer": layer,
                            "feature": feature,
                            "role": feature_info["role"],
                            "candidate_layer": feature_info["candidate_layer"],
                            "candidate_feature": feature_info["candidate_feature"],
                            "candidate_tier": feature_info["candidate_tier"],
                            "activation": activation,
                            "token_position": pos,
                            "window_index": window_idx,
                            "window_start": window["window_start"],
                            "context": context,
                            "window_text": window["window_text"],
                            "input_ids_json": json.dumps(ids)
                        })

            del outputs, input_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Save a checkpoint after each corpus.
    pd.DataFrame(all_contexts).to_csv(
        f"{OUT}/{corpus_name}_contexts.csv",
        index=False
    )

contexts = pd.DataFrame(all_contexts)

if contexts.empty:
    raise RuntimeError("No feature activations were extracted.")

contexts.to_csv(f"{OUT}/all_contexts.csv", index=False)

print("\nTotal extracted contexts:", len(contexts))
print("Active feature combinations:", contexts[
    ["layer", "feature", "role", "corpus"]
].drop_duplicates().shape[0])

# ----------------------- CONTEXT DEDUPLICATION ---------------------

contexts["context_norm"] = (
    contexts["context"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

contexts = contexts.drop_duplicates(
    ["corpus", "layer", "feature", "context_norm"]
).copy()

# ----------------------- SEMANTIC EMBEDDINGS -----------------------

print("\nLoading sentence embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE
)

contexts["embedding_index"] = -1

embedding_cache = {}

unique_texts = contexts["context"].fillna("").tolist()

embeddings = embedder.encode(
    unique_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

contexts["embedding_index"] = np.arange(len(contexts))

# ----------------------- CLUSTERING HELPERS ------------------------

def cluster_terms(texts, labels, top_n=8):
    output = {}

    for cluster_id in sorted(set(labels)):
        selected_texts = [
            texts[i] for i in range(len(texts))
            if labels[i] == cluster_id
        ]

        if len(selected_texts) < 2:
            output[int(cluster_id)] = []
            continue

        try:
            vectorizer = TfidfVectorizer(
                stop_words="english",
                ngram_range=(1, 2),
                max_features=3000,
                min_df=1
            )

            matrix = vectorizer.fit_transform(selected_texts)
            scores = np.asarray(matrix.mean(axis=0)).ravel()
            terms = np.array(vectorizer.get_feature_names_out())

            top = np.argsort(scores)[::-1][:top_n]
            output[int(cluster_id)] = terms[top].tolist()

        except Exception:
            output[int(cluster_id)] = []

    return output


def run_feature_clustering(group):

    group = group.reset_index(drop=True)

    if len(group) < MIN_CONTEXTS:
        return {
            "evaluable": False,
            "n_contexts": len(group)
        }

    idx = group["embedding_index"].astype(int).values
    X = embeddings[idx]

    texts = group["context"].astype(str).tolist()

    best = None

    for k in range(2, min(MAX_CLUSTER_K, len(group) // 5) + 1):

        labels_all = []

        for seed in KMEANS_SEEDS:
            km = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=10
            )

            labels = km.fit_predict(X)
            labels_all.append(labels)

        sils = []

        for labels in labels_all:
            if len(set(labels)) > 1:
                sils.append(
                    silhouette_score(X, labels, metric="cosine")
                )

        if not sils:
            continue

        # Use the first seed for cluster interpretation.
        labels = labels_all[0]

        stability = []

        for other in labels_all[1:]:
            stability.append(
                adjusted_rand_score(labels, other)
            )

        candidate = {
            "k": k,
            "silhouette": float(np.mean(sils)),
            "silhouette_sd": float(np.std(sils)),
            "seed_stability": float(np.mean(stability)),
            "labels": labels
        }

        if best is None or candidate["silhouette"] > best["silhouette"]:
            best = candidate

    if best is None:
        return {
            "evaluable": False,
            "n_contexts": len(group)
        }

    # Bootstrap stability: compare labels on sampled observations.
    bootstrap_scores = []

    rng = np.random.default_rng(SEED)
    base_labels = best["labels"]
    k = best["k"]

    for _ in range(BOOTSTRAP_RUNS):

        sample_idx = rng.choice(
            len(X),
            size=max(10, int(0.8 * len(X))),
            replace=False
        )

        Xb = X[sample_idx]

        try:
            km = KMeans(
                n_clusters=k,
                random_state=int(rng.integers(1, 1_000_000)),
                n_init=10
            )

            lb = km.fit_predict(Xb)

            bootstrap_scores.append(
                adjusted_rand_score(
                    base_labels[sample_idx],
                    lb
                )
            )
        except Exception:
            pass

    terms = cluster_terms(texts, best["labels"])

    sizes = {
        str(int(c)): int(np.sum(best["labels"] == c))
        for c in sorted(set(best["labels"]))
    }

    return {
        "evaluable": True,
        "n_contexts": len(group),
        "n_documents": group["document_id"].nunique(),
        "k": best["k"],
        "silhouette": best["silhouette"],
        "silhouette_sd": best["silhouette_sd"],
        "seed_stability": best["seed_stability"],
        "bootstrap_stability": (
            float(np.mean(bootstrap_scores))
            if bootstrap_scores else np.nan
        ),
        "cluster_sizes": json.dumps(sizes),
        "cluster_terms": json.dumps(terms),
        "labels": best["labels"]
    }


# ----------------------- RUN ALL CLUSTER ANALYSES -----------------

metric_rows = []
cluster_context_rows = []

grouped = contexts.groupby(
    ["corpus", "layer", "feature", "role"],
    sort=False
)

for (corpus, layer, feature, role), group in tqdm(
    grouped,
    total=grouped.ngroups,
    desc="Clustering features"
):

    result = run_feature_clustering(group)

    row = {
        "corpus": corpus,
        "layer": int(layer),
        "feature": int(feature),
        "role": role,
        "evaluable": result["evaluable"],
        "n_contexts": result["n_contexts"]
    }

    for key in [
        "n_documents", "k", "silhouette", "silhouette_sd",
        "seed_stability", "bootstrap_stability",
        "cluster_sizes", "cluster_terms"
    ]:
        row[key] = result.get(key, np.nan)

    metric_rows.append(row)

    if result["evaluable"]:
        temp = group.reset_index(drop=True).copy()
        temp["cluster"] = result["labels"]
        cluster_context_rows.append(temp)

metrics = pd.DataFrame(metric_rows)

metrics.to_csv(f"{OUT}/validation_metrics.csv", index=False)

if cluster_context_rows:
    clustered_contexts = pd.concat(
        cluster_context_rows,
        ignore_index=True
    )
else:
    clustered_contexts = pd.DataFrame()

clustered_contexts.to_csv(
    f"{OUT}/validated_context_clusters.csv",
    index=False
)

print("\nClustering complete.")
print(metrics.groupby(["role", "corpus"])["evaluable"].mean())

# ----------------------- PAIRED CANDIDATE-CONTROL ANALYSIS ---------

# Each candidate is compared with its own prevalence-matched control
# in the same layer and corpus.

pair_rows = []

for corpus in corpora.keys():

    cand = metrics[
        (metrics.corpus == corpus) &
        (metrics.role == "candidate") &
        (metrics.evaluable == True)
    ]

    ctrl = metrics[
        (metrics.corpus == corpus) &
        (metrics.role == "matched_control") &
        (metrics.evaluable == True)
    ]

    for _, c in cand.iterrows():

        matched = control_df[
            (control_df.candidate_layer == c.layer) &
            (control_df.candidate_feature == c.feature)
        ]

        if matched.empty:
            continue

        control_feature = int(matched.iloc[0].feature)

        cr = ctrl[
            (ctrl.layer == c.layer) &
            (ctrl.feature == control_feature)
        ]

        if cr.empty:
            continue

        cr = cr.iloc[0]

        pair_rows.append({
            "corpus": corpus,
            "layer": int(c.layer),
            "candidate_feature": int(c.feature),
            "control_feature": control_feature,
            "candidate_silhouette": float(c.silhouette),
            "control_silhouette": float(cr.silhouette),
            "silhouette_difference": float(
                c.silhouette - cr.silhouette
            ),
            "candidate_stability": float(c.seed_stability),
            "control_stability": float(cr.seed_stability),
            "candidate_contexts": int(c.n_contexts),
            "control_contexts": int(cr.n_contexts)
        })

paired = pd.DataFrame(pair_rows)
paired.to_csv(f"{OUT}/paired_candidate_control.csv", index=False)

# Paired sign-flip permutation test across candidate-control pairs.
def paired_sign_flip_test(values, runs=PERMUTATION_RUNS):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 5:
        return np.nan, np.nan, np.nan

    observed = float(np.mean(values))

    rng = np.random.default_rng(SEED)
    null = []

    for _ in range(runs):
        signs = rng.choice([-1, 1], size=len(values))
        null.append(float(np.mean(values * signs)))

    null = np.asarray(null)

    p = (
        1 + np.sum(np.abs(null) >= abs(observed))
    ) / (runs + 1)

    return observed, float(p), float(np.std(null))


group_tests = []

for corpus, group in paired.groupby("corpus"):

    mean_diff, p, null_sd = paired_sign_flip_test(
        group["silhouette_difference"]
    )

    group_tests.append({
        "corpus": corpus,
        "n_pairs": len(group),
        "mean_candidate_minus_control_silhouette": mean_diff,
        "permutation_p": p,
        "null_sd": null_sd
    })

group_tests = pd.DataFrame(group_tests)
group_tests.to_csv(f"{OUT}/group_level_permutation_tests.csv", index=False)

# ----------------------- FEATURE-LEVEL BOOTSTRAP ------------------

# Bootstrap confidence intervals are descriptive, not a replacement
# for independent-document replication.

rng = np.random.default_rng(SEED)
feature_bootstrap_rows = []

for (corpus, layer, feature, role), group in contexts.groupby(
    ["corpus", "layer", "feature", "role"]
):

    vals = group["activation"].astype(float).values

    if len(vals) < MIN_CONTEXTS:
        continue

    boot_means = []

    for _ in range(500):
        sample = rng.choice(vals, len(vals), replace=True)
        boot_means.append(np.mean(sample))

    feature_bootstrap_rows.append({
        "corpus": corpus,
        "layer": int(layer),
        "feature": int(feature),
        "role": role,
        "n_contexts": len(vals),
        "activation_mean": float(np.mean(vals)),
        "activation_ci_low": float(np.percentile(boot_means, 2.5)),
        "activation_ci_high": float(np.percentile(boot_means, 97.5))
    })

pd.DataFrame(feature_bootstrap_rows).to_csv(
    f"{OUT}/activation_bootstrap.csv",
    index=False
)

# ----------------------- FDR CORRECTION ----------------------------

# The paired permutation tests above are corpus-level tests.
# Correct their p-values across the tested corpora.

if not group_tests.empty:
    valid = group_tests["permutation_p"].notna()

    group_tests["fdr_q"] = np.nan

    if valid.sum():
        group_tests.loc[valid, "fdr_q"] = multipletests(
            group_tests.loc[valid, "permutation_p"],
            method="fdr_bh"
        )[1]

    group_tests.to_csv(
        f"{OUT}/group_level_permutation_tests.csv",
        index=False
    )

# ----------------------- CROSS-CORPUS REPLICATION -----------------

replication_rows = []

for _, cand in candidate_df.iterrows():

    layer = int(cand.layer)
    feature = int(cand.feature)

    a = metrics[
        (metrics.corpus == "wikitext_test") &
        (metrics.layer == layer) &
        (metrics.feature == feature) &
        (metrics.role == "candidate")
    ]

    b = metrics[
        (metrics.corpus == "c4_validation") &
        (metrics.layer == layer) &
        (metrics.feature == feature) &
        (metrics.role == "candidate")
    ]

    if a.empty or b.empty:
        continue

    a, b = a.iloc[0], b.iloc[0]

    replication_rows.append({
        "layer": layer,
        "feature": feature,
        "wikitext_evaluable": bool(a.evaluable),
        "c4_evaluable": bool(b.evaluable),
        "wikitext_silhouette": a.silhouette,
        "c4_silhouette": b.silhouette,
        "wikitext_stability": a.seed_stability,
        "c4_stability": b.seed_stability,
        "replicated_semantic_structure": bool(
            a.evaluable and b.evaluable and
            a.silhouette >= 0.15 and
            b.silhouette >= 0.15 and
            a.seed_stability >= 0.50 and
            b.seed_stability >= 0.50
        )
    })

replication = pd.DataFrame(replication_rows)

replication.to_csv(
    f"{OUT}/cross_corpus_replication.csv",
    index=False
)

# ----------------------- SELECT CAUSAL CANDIDATES ------------------

# Prioritize features with enough contexts and the strongest
# cross-corpus semantic structure. This selection is exploratory.

candidate_metrics = metrics[
    (metrics.role == "candidate") &
    (metrics.evaluable == True)
].copy()

wide = candidate_metrics.pivot_table(
    index=["layer", "feature"],
    columns="corpus",
    values="silhouette",
    aggfunc="first"
).reset_index()

wide = wide.merge(
    replication,
    on=["layer", "feature"],
    how="left"
)

wide["causal_priority"] = (
    wide.get("wikitext_test", pd.Series(index=wide.index, dtype=float))
    .fillna(0) +
    wide.get("c4_validation", pd.Series(index=wide.index, dtype=float))
    .fillna(0)
)

causal_features = wide.sort_values(
    "causal_priority",
    ascending=False
).head(CAUSAL_FEATURE_LIMIT)

causal_features.to_csv(
    f"{OUT}/causal_feature_selection.csv",
    index=False
)

# ----------------------- CAUSAL FEATURE ABLATION -------------------

# For each selected feature:
# 1. Retrieve a high-activation context.
# 2. Re-run the exact tokenized window.
# 3. Capture the corresponding post-block residual state.
# 4. Compute the feature activation using the existing SAE.
# 5. Subtract that feature's decoder contribution at the selected token.
# 6. Measure the change in next-token distribution.
#
# This tests the causal effect of removing one SAE feature contribution.
# It does not establish that the feature is the only cause of the behavior.

def get_block_output(output):
    if isinstance(output, tuple):
        return output[0]
    return output


def causal_ablation(layer, feature, input_ids, token_position):

    sae = saes[layer]

    # Baseline forward pass and capture of the selected block output.
    captured = {}

    block = model.model.layers[layer - 1]

    def capture_hook(module, inputs, output):
        captured["hidden"] = get_block_output(output).detach()

    handle = block.register_forward_hook(capture_hook)

    with torch.inference_mode():
        baseline = model(
            input_ids=input_ids,
            use_cache=False
        )

    handle.remove()

    baseline_logits = baseline.logits[0, -1].float()
    hidden = captured["hidden"][0]

    if token_position >= hidden.shape[0]:
        return None

    x = hidden[token_position].to(torch.float16)
    centered = x - sae["mean"]

    pre = F.linear(
        centered,
        sae["W_enc"],
        sae["b_enc"]
    )

    pre = F.relu(pre)

    top_values, top_indices = torch.topk(
        pre, k=TOP_K, dim=-1
    )

    matches = torch.where(top_indices == feature)[0]

    if len(matches) == 0:
        return {
            "feature_active": False,
            "activation": 0.0
        }

    activation = top_values[matches[0]]

    # Decoder vector for this feature, in residual-stream coordinates.
    decoder_direction = sae["W_dec"][feature].to(torch.float16)

    delta = activation * decoder_direction

    # Ablate only the selected token position.
    def ablation_hook(module, inputs, output):
        tensor = get_block_output(output)
        modified = tensor.clone()
        modified[:, token_position, :] -= delta.to(
            modified.device, dtype=modified.dtype
        )

        if isinstance(output, tuple):
            return (modified,) + output[1:]

        return modified

    handle = block.register_forward_hook(ablation_hook)

    with torch.inference_mode():
        ablated = model(
            input_ids=input_ids,
            use_cache=False
        )

    handle.remove()

    ablated_logits = ablated.logits[0, -1].float()

    p_base = F.softmax(baseline_logits, dim=-1)
    p_ablated = F.softmax(ablated_logits, dim=-1)

    eps = 1e-12

    kl = torch.sum(
        p_base * (
            torch.log(p_base + eps) -
            torch.log(p_ablated + eps)
        )
    )

    js_mid = 0.5 * (p_base + p_ablated)

    js = 0.5 * torch.sum(
        p_base * (
            torch.log(p_base + eps) -
            torch.log(js_mid + eps)
        )
    ) + 0.5 * torch.sum(
        p_ablated * (
            torch.log(p_ablated + eps) -
            torch.log(js_mid + eps)
        )
    )

    top_base = torch.topk(baseline_logits, 5).indices.tolist()
    top_ablated = torch.topk(ablated_logits, 5).indices.tolist()

    return {
        "feature_active": True,
        "activation": float(activation.float().item()),
        "kl_baseline_to_ablated": float(kl.item()),
        "jensen_shannon": float(js.item()),
        "logit_l2_change": float(
            torch.norm(baseline_logits - ablated_logits).item()
        ),
        "top5_baseline": json.dumps(top_base),
        "top5_ablated": json.dumps(top_ablated),
        "top1_changed": bool(top_base[0] != top_ablated[0])
    }


causal_rows = []

for _, selected_feature in tqdm(
    causal_features.iterrows(),
    total=len(causal_features),
    desc="Causal ablations"
):

    layer = int(selected_feature.layer)
    feature = int(selected_feature.feature)

    feature_contexts = contexts[
        (contexts.layer == layer) &
        (contexts.feature == feature) &
        (contexts.role == "candidate")
    ].sort_values("activation", ascending=False)

    # Prefer contexts from both corpora.
    chosen = []

    for corpus_name in ["wikitext_test", "c4_validation"]:
        subset = feature_contexts[
            feature_contexts.corpus == corpus_name
        ].head(2)

        chosen.extend([
            r for _, r in subset.iterrows()
        ])

    chosen = chosen[:CAUSAL_CONTEXTS_PER_FEATURE]

    for row in chosen:

        ids = json.loads(row.input_ids_json)

        input_ids = torch.tensor(
            [ids], device=DEVICE, dtype=torch.long
        )

        try:
            result = causal_ablation(
                layer,
                feature,
                input_ids,
                int(row.token_position)
            )

            if result is None:
                continue

            causal_rows.append({
                "corpus": row.corpus,
                "document_id": row.document_id,
                "layer": layer,
                "feature": feature,
                "token_position": int(row.token_position),
                "context": row.context,
                **result
            })

        except Exception as e:
            causal_rows.append({
                "corpus": row.corpus,
                "document_id": row.document_id,
                "layer": layer,
                "feature": feature,
                "context": row.context,
                "error": str(e)
            })

        del input_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

causal_df = pd.DataFrame(causal_rows)

causal_df.to_csv(
    f"{OUT}/causal_ablation_results.csv",
    index=False
)

# ----------------------- CONTROL ABLATION --------------------------

# Run the same intervention for matched controls.
# This is important: a large logit change is not specific evidence
# unless candidate features differ from comparable controls.

control_causal_rows = []

for _, selected_feature in tqdm(
    causal_features.iterrows(),
    total=len(causal_features),
    desc="Matched-control ablations"
):

    layer = int(selected_feature.layer)
    feature = int(selected_feature.feature)

    match = control_df[
        (control_df.candidate_layer == layer) &
        (control_df.candidate_feature == feature)
    ]

    if match.empty:
        continue

    control_feature = int(match.iloc[0].feature)

    control_contexts = contexts[
        (contexts.layer == layer) &
        (contexts.feature == control_feature) &
        (contexts.role == "matched_control")
    ].sort_values("activation", ascending=False).head(2)

    for _, row in control_contexts.iterrows():

        ids = json.loads(row.input_ids_json)

        input_ids = torch.tensor(
            [ids], device=DEVICE, dtype=torch.long
        )

        try:
            result = causal_ablation(
                layer,
                control_feature,
                input_ids,
                int(row.token_position)
            )

            if result is None:
                continue

            control_causal_rows.append({
                "corpus": row.corpus,
                "document_id": row.document_id,
                "layer": layer,
                "candidate_feature": feature,
                "control_feature": control_feature,
                "context": row.context,
                **result
            })

        except Exception as e:
            control_causal_rows.append({
                "corpus": row.corpus,
                "layer": layer,
                "candidate_feature": feature,
                "control_feature": control_feature,
                "error": str(e)
            })

        del input_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

control_causal_df = pd.DataFrame(control_causal_rows)

control_causal_df.to_csv(
    f"{OUT}/matched_control_ablation_results.csv",
    index=False
)

# ----------------------- CAUSAL SUMMARY ----------------------------

def safe_mean(df, column):
    if df.empty or column not in df:
        return np.nan
    return pd.to_numeric(df[column], errors="coerce").mean()


causal_summary = {
    "candidate_interventions": len(causal_df),
    "control_interventions": len(control_causal_df),
    "candidate_active_interventions": int(
        causal_df.get(
            "feature_active", pd.Series(dtype=bool)
        ).fillna(False).sum()
    ),
    "control_active_interventions": int(
        control_causal_df.get(
            "feature_active", pd.Series(dtype=bool)
        ).fillna(False).sum()
    ),
    "candidate_mean_js": safe_mean(causal_df, "jensen_shannon"),
    "control_mean_js": safe_mean(control_causal_df, "jensen_shannon"),
    "candidate_mean_kl": safe_mean(
        causal_df, "kl_baseline_to_ablated"
    ),
    "control_mean_kl": safe_mean(
        control_causal_df, "kl_baseline_to_ablated"
    ),
    "candidate_top1_change_rate": safe_mean(
        causal_df, "top1_changed"
    ),
    "control_top1_change_rate": safe_mean(
        control_causal_df, "top1_changed"
    )
}

with open(f"{OUT}/causal_summary.json", "w") as f:
    json.dump(causal_summary, f, indent=2)

# ----------------------- FINAL SUMMARY -----------------------------

candidate_pass = metrics[
    (metrics.role == "candidate") &
    (metrics.evaluable == True)
]

control_pass = metrics[
    (metrics.role == "matched_control") &
    (metrics.evaluable == True)
]

summary = {
    "experiment": "Experiment 03B - Corrected Polysemanticity Validation",
    "run_id": RUN_ID,
    "model": MODEL_NAME,
    "revision": MODEL_REVISION,
    "sae_checkpoints_reused": len(saes),
    "sae_architecture": {
        "input_dimension": D_MODEL,
        "latent_dimension": D_SAE,
        "top_k": TOP_K
    },
    "candidate_count": len(candidate_df),
    "matched_control_count": len(control_df),
    "wikitext_articles": len(corpora["wikitext_test"]),
    "c4_documents": len(corpora["c4_validation"]),
    "total_contexts": len(contexts),
    "candidate_evaluable_feature_corpus_pairs": len(candidate_pass),
    "control_evaluable_feature_corpus_pairs": len(control_pass),
    "candidate_mean_silhouette": safe_mean(
        candidate_pass, "silhouette"
    ),
    "control_mean_silhouette": safe_mean(
        control_pass, "silhouette"
    ),
    "candidate_mean_seed_stability": safe_mean(
        candidate_pass, "seed_stability"
    ),
    "control_mean_seed_stability": safe_mean(
        control_pass, "seed_stability"
    ),
    "cross_corpus_replicated_candidates": int(
        replication.get(
            "replicated_semantic_structure",
            pd.Series(dtype=bool)
        ).fillna(False).sum()
    ),
    "causal_summary": causal_summary,
    "methodological_note": (
        "Semantic clustering and ablation are complementary evidence. "
        "Ablation effects can include collateral effects and do not alone "
        "prove a uniquely interpretable feature. Interpret results against "
        "matched controls and inspect contexts."
    )
}

with open(f"{OUT}/experiment_03B_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ----------------------- EXPORT ZIP --------------------------------

zip_path = f"{PROJECT_DIR}/Experiment_03B_Corrected_{RUN_ID}.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:

    for path in glob.glob(OUT + "/**/*", recursive=True):
        if os.path.isfile(path):
            z.write(
                path,
                arcname=os.path.relpath(path, OUT)
            )

# ----------------------- FINAL REPORT ------------------------------

print("\n" + "=" * 65)
print("EXPERIMENT 03B COMPLETE")
print("=" * 65)

print("\nSUMMARY")
print(json.dumps(summary, indent=2))

print("\nGROUP-LEVEL PAIRED TESTS")
display(group_tests)

print("\nCROSS-CORPUS REPLICATION")
display(
    replication.sort_values(
        ["replicated_semantic_structure", "wikitext_silhouette"],
        ascending=False
    ).head(20)
)

print("\nCAUSAL ABLATION RESULTS")
display(causal_df.head(20))

print("\nMATCHED CONTROL ABLATION RESULTS")
display(control_causal_df.head(20))

print("\nOUTPUT DIRECTORY:")
print(OUT)

print("\nZIP ARCHIVE:")
print(zip_path)

print("\nDownload or retrieve the ZIP from Google Drive.")
print("No SAE retraining was performed.")


In [ ]:
from google.colab import files

zip_path = "/content/drive/MyDrive/Mapping_Feature_Drift/sae_research_100k_v2/Experiment_03B_Corrected_20261110_063542.zip"

files.download(zip_path)

In [ ]:

# ================================================================
# TARGETED FORENSIC ANALYSIS: LAYER 33, FEATURE 374
# Candidate: L33:F374
# Matched control: L33:F3143
# ================================================================

import os, json, zipfile, random, re
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import TfidfVectorizer

from sentence_transformers import SentenceTransformer

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

ZIP_PATH = (
    "/content/drive/MyDrive/Mapping_Feature_Drift/"
    "sae_research_100k_v2/"
    "Experiment_03B_Corrected_20261110_063542.zip"
)

LAYER = 33
CANDIDATE = 374
CONTROL = 3143

OUT = (
    "/content/drive/MyDrive/Mapping_Feature_Drift/"
    "sae_research_100k_v2/"
    "experiment_03B_corrected/"
    "feature_33_374_forensic"
)

os.makedirs(OUT, exist_ok=True)

assert os.path.isfile(ZIP_PATH), f"ZIP not found: {ZIP_PATH}"

# ------------------------------------------------
# 1. READ ARCHIVE
# ------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    names = z.namelist()
    print("Archive files:", names)

    def read_csv(name):
        matches = [
            n for n in names
            if n.endswith(name)
        ]

        if not matches:
            print("Missing:", name)
            return pd.DataFrame()

        with z.open(matches[0]) as f:
            return pd.read_csv(f)

    contexts = read_csv("all_contexts.csv")
    metrics = read_csv("validation_metrics.csv")
    causal = read_csv("causal_ablation_results.csv")
    control_causal = read_csv(
        "matched_control_ablation_results.csv"
    )
    replication = read_csv("cross_corpus_replication.csv")

assert not contexts.empty, "Could not load all_contexts.csv"

# ------------------------------------------------
# 2. FILTER TARGET FEATURE AND CONTROL
# ------------------------------------------------

candidate = contexts[
    (contexts["layer"] == LAYER) &
    (contexts["feature"] == CANDIDATE) &
    (contexts["role"] == "candidate")
].copy()

control = contexts[
    (contexts["layer"] == LAYER) &
    (contexts["feature"] == CONTROL) &
    (contexts["role"] == "matched_control")
].copy()

print("\nCandidate contexts:", len(candidate))
print("Control contexts:", len(control))

print("\nCandidate documents by corpus:")
print(candidate.groupby("corpus")["document_id"].nunique())

print("\nControl documents by corpus:")
print(control.groupby("corpus")["document_id"].nunique())

# Remove repeated contexts.
for df in [candidate, control]:
    df["normalized_context"] = (
        df["context"].fillna("")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

candidate = candidate.drop_duplicates(
    ["corpus", "document_id", "normalized_context"]
)

control = control.drop_duplicates(
    ["corpus", "document_id", "normalized_context"]
)

# ------------------------------------------------
# 3. PRINT ACTIVATION DISTRIBUTION
# ------------------------------------------------

def activation_summary(df, name):
    print(f"\n{name}")

    if df.empty:
        print("No contexts.")
        return

    print(
        df.groupby("corpus")["activation"]
        .agg(["count", "mean", "median", "min", "max"])
    )

activation_summary(candidate, "L33:F374")
activation_summary(control, "L33:F3143")

# ------------------------------------------------
# 4. PRINT STRONGEST AND RANDOM CONTEXTS
# ------------------------------------------------

def show_contexts(df, name, n=20):

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    if df.empty:
        print("No contexts.")
        return

    for corpus_name in sorted(df["corpus"].unique()):

        sub = df[df["corpus"] == corpus_name]

        print(f"\n--- {corpus_name}: strongest activations ---")

        strongest = sub.nlargest(
            min(n, len(sub)), "activation"
        )

        for i, row in enumerate(strongest.itertuples(), 1):
            print(
                f"\n{i}. Activation={row.activation:.4f} | "
                f"Document={row.document_id}"
            )
            print(row.context)

        print(f"\n--- {corpus_name}: random contexts ---")

        sample = sub.sample(
            min(n, len(sub)),
            random_state=SEED
        )

        for i, row in enumerate(sample.itertuples(), 1):
            print(
                f"\n{i}. Activation={row.activation:.4f} | "
                f"Document={row.document_id}"
            )
            print(row.context)


show_contexts(candidate, "CANDIDATE L33:F374")
show_contexts(control, "MATCHED CONTROL L33:F3143")

# ------------------------------------------------
# 5. EXISTING CAUSAL EVIDENCE
# ------------------------------------------------

print("\n" + "=" * 90)
print("EXISTING CAUSAL ABLATION: L33:F374")
print("=" * 90)

if not causal.empty:
    target_causal = causal[
        (causal["layer"] == LAYER) &
        (causal["feature"] == CANDIDATE)
    ]

    display_cols = [
        c for c in [
            "corpus", "document_id", "token_position",
            "activation", "kl_baseline_to_ablated",
            "jensen_shannon", "logit_l2_change",
            "top1_changed", "context"
        ] if c in target_causal.columns
    ]

    print(target_causal[display_cols].to_string(index=False))

print("\nMatched control ablations:")

if not control_causal.empty:
    target_control = control_causal[
        (control_causal["layer"] == LAYER) &
        (control_causal["candidate_feature"] == CANDIDATE)
    ]

    display_cols = [
        c for c in [
            "corpus", "document_id", "activation",
            "kl_baseline_to_ablated", "jensen_shannon",
            "logit_l2_change", "top1_changed", "context"
        ] if c in target_control.columns
    ]

    print(target_control[display_cols].to_string(index=False))

# ------------------------------------------------
# 6. SEMANTIC EMBEDDING ANALYSIS
# ------------------------------------------------

print("\nLoading embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def evaluate_semantic_structure(df, name):

    print("\n" + "=" * 90)
    print("SEMANTIC ANALYSIS:", name)
    print("=" * 90)

    if len(df) < 20:
        print("Insufficient contexts.")
        return None

    texts = df["context"].astype(str).tolist()

    X = embedder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    results = []
    best = None

    for k in range(2, 7):

        if len(df) < k * 8:
            continue

        seed_labels = []

        for seed in [11, 29, 47, 83, 101]:

            km = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=20
            )

            labels = km.fit_predict(X)
            seed_labels.append(labels)

        silhouettes = [
            silhouette_score(X, labels, metric="cosine")
            for labels in seed_labels
        ]

        stability = np.mean([
            adjusted_rand_score(
                seed_labels[0], labels
            )
            for labels in seed_labels[1:]
        ])

        labels = seed_labels[0]

        sizes = Counter(labels)

        result = {
            "feature": name,
            "k": k,
            "silhouette_mean": float(np.mean(silhouettes)),
            "silhouette_sd": float(np.std(silhouettes)),
            "seed_stability": float(stability),
            "cluster_sizes": json.dumps(dict(sizes))
        }

        results.append(result)

        if (
            best is None or
            result["silhouette_mean"] >
            best["silhouette_mean"]
        ):
            best = {
                **result,
                "labels": labels,
                "embeddings": X
            }

    results_df = pd.DataFrame(results)

    print("\nClustering results:")
    display(results_df)

    if best is not None:

        df = df.reset_index(drop=True).copy()
        df["semantic_cluster"] = best["labels"]

        print("\nBest clustering:", best["k"], "clusters")

        for cluster_id in sorted(df.semantic_cluster.unique()):

            subset = df[
                df.semantic_cluster == cluster_id
            ]

            print("\n" + "-" * 70)
            print(
                f"CLUSTER {cluster_id} | "
                f"{len(subset)} contexts | "
                f"{subset.document_id.nunique()} documents"
            )

            print(
                "Corpora:",
                subset["corpus"].value_counts().to_dict()
            )

            print("Examples:")

            for row in subset.nlargest(
                min(8, len(subset)), "activation"
            ).itertuples():

                print(
                    f"\nActivation {row.activation:.3f}: "
                    f"{row.context}"
                )

        return results_df, df

    return results_df, df


candidate_analysis = evaluate_semantic_structure(
    candidate,
    "L33:F374"
)

control_analysis = evaluate_semantic_structure(
    control,
    "L33:F3143"
)

# ------------------------------------------------
# 7. ACTIVATION-STRATIFIED SEMANTIC CHECK
# ------------------------------------------------

def activation_strata(df, name):

    if df.empty:
        return

    print("\n" + "=" * 90)
    print("ACTIVATION STRATA:", name)
    print("=" * 90)

    df = df.copy()

    df["activation_band"] = pd.qcut(
        df["activation"].rank(method="first"),
        q=min(4, len(df)),
        labels=False
    )

    for band, subset in df.groupby("activation_band"):

        print(
            f"\nActivation band {band}: "
            f"{len(subset)} contexts"
        )

        for row in subset.sample(
            min(8, len(subset)),
            random_state=SEED
        ).itertuples():

            print("-", row.context)

activation_strata(candidate, "L33:F374")

# ------------------------------------------------
# 8. SAVE FORENSIC REPORT
# ------------------------------------------------

candidate.to_csv(
    f"{OUT}/candidate_contexts.csv",
    index=False
)

control.to_csv(
    f"{OUT}/matched_control_contexts.csv",
    index=False
)

if candidate_analysis:
    candidate_analysis[1].to_csv(
        f"{OUT}/candidate_semantic_clusters.csv",
        index=False
    )
    candidate_analysis[0].to_csv(
        f"{OUT}/candidate_cluster_metrics.csv",
        index=False
    )

if control_analysis:
    control_analysis[1].to_csv(
        f"{OUT}/control_semantic_clusters.csv",
        index=False
    )
    control_analysis[0].to_csv(
        f"{OUT}/control_cluster_metrics.csv",
        index=False
    )

print("\nSaved forensic outputs to:")
print(OUT)


In [ ]:

# ============================================================
# RESUME L33:F374 FORENSIC ANALYSIS
# ============================================================

import json
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

def evaluate_semantic_structure_fixed(df, name):

    print("\n" + "=" * 80)
    print("SEMANTIC ANALYSIS:", name)
    print("=" * 80)

    if len(df) < 20:
        print("Insufficient contexts.")
        return None

    texts = df["context"].astype(str).tolist()

    X = embedder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    results = []
    best = None

    for k in range(2, 7):

        if len(df) < k * 8:
            continue

        seed_labels = []

        for seed in [11, 29, 47, 83, 101]:

            km = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=20
            )

            labels = km.fit_predict(X)
            seed_labels.append(labels)

        silhouettes = [
            silhouette_score(X, labels, metric="cosine")
            for labels in seed_labels
        ]

        stability = np.mean([
            adjusted_rand_score(
                seed_labels[0], labels
            )
            for labels in seed_labels[1:]
        ])

        labels = seed_labels[0]

        sizes = Counter(int(x) for x in labels)

        result = {
            "feature": name,
            "k": int(k),
            "silhouette_mean": float(np.mean(silhouettes)),
            "silhouette_sd": float(np.std(silhouettes)),
            "seed_stability": float(stability),
            "cluster_sizes": json.dumps({
                int(key): int(value)
                for key, value in sizes.items()
            })
        }

        results.append(result)

        if (
            best is None or
            result["silhouette_mean"] >
            best["silhouette_mean"]
        ):
            best = {
                **result,
                "labels": labels
            }

    results_df = pd.DataFrame(results)

    print("\nClustering metrics:")
    display(results_df)

    clustered = df.reset_index(drop=True).copy()

    if best is not None:

        clustered["semantic_cluster"] = best["labels"]

        print("\nBest K:", best["k"])

        for cluster_id in sorted(
            clustered["semantic_cluster"].unique()
        ):

            subset = clustered[
                clustered["semantic_cluster"] == cluster_id
            ]

            print("\n" + "-" * 70)
            print(
                f"CLUSTER {cluster_id} | "
                f"{len(subset)} contexts | "
                f"{subset['document_id'].nunique()} documents"
            )

            print(
                "Corpus distribution:",
                subset["corpus"].value_counts().to_dict()
            )

            for row in subset.nlargest(
                min(10, len(subset)), "activation"
            ).itertuples():

                print(
                    f"\nActivation={row.activation:.3f}"
                )
                print(row.context)

    return results_df, clustered


# Candidate
candidate_analysis = evaluate_semantic_structure_fixed(
    candidate,
    "L33:F374"
)

# Matched control
control_analysis = evaluate_semantic_structure_fixed(
    control,
    "L33:F3143"
)


# Save results
OUT = (
    "/content/drive/MyDrive/Mapping_Feature_Drift/"
    "sae_research_100k_v2/"
    "experiment_03B_corrected/"
    "feature_33_374_forensic"
)

import os
os.makedirs(OUT, exist_ok=True)

candidate_analysis[0].to_csv(
    f"{OUT}/candidate_cluster_metrics.csv",
    index=False
)

candidate_analysis[1].to_csv(
    f"{OUT}/candidate_semantic_clusters.csv",
    index=False
)

control_analysis[0].to_csv(
    f"{OUT}/control_cluster_metrics.csv",
    index=False
)

control_analysis[1].to_csv(
    f"{OUT}/control_semantic_clusters.csv",
    index=False
)

print("\nAnalysis completed. Results saved to:")
print(OUT)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ============================================================
# L33:F374 — TARGETED POLYSEMANTICITY INVESTIGATION
# Qwen2.5-3B + Existing Layer 33 SAE
# ============================================================

# ============================================================
# 0. ENVIRONMENT
# ============================================================

import os
import gc
import json
import torch
import torch.nn.functional as F
import transformers

from transformers import AutoModel, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 85)
print("ENVIRONMENT")
print("=" * 85)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

assert torch.cuda.is_available(), (
    "GPU is required. Select a GPU runtime in Colab."
)

print("GPU:", torch.cuda.get_device_name(0))

torch.cuda.empty_cache()


# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B"
REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"

LAYER = 33
FEATURE = 374

INPUT_DIM = 2048
LATENT_DIM = 8192
TOP_K = 32
MAX_LENGTH = 256

SAE_PATH = "/content/sae_layer_33.pt"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Mapping_Feature_Drift/"
    "experiment_03B_corrected/targeted_L33_F374"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert 0 <= FEATURE < LATENT_DIM
assert 0 <= LAYER < 36


# ============================================================
# 2. TEST EXAMPLES
# ============================================================

examples = {
    "Christian eschatology": (
        "The Sibylline Oracles, Book 5 and 8, written in the "
        "2nd century, speak of Nero returning and bringing destruction. "
        "Within Christian communities, these writings fueled the belief "
        "that Nero would return as the Antichrist."
    ),

    "Chinese philosophy and literature": (
        "The poetry of Wang Wei, Li Bai and Du Fu reflects the influence "
        "of Buddhist, Daoist and Confucian philosophical traditions "
        "in Chinese literary history."
    ),

    "Jewish religious institutions": (
        "Jewish religious institutions developed educational traditions "
        "that preserved religious texts, historical knowledge and "
        "community practices across generations."
    ),

    "Christian institutional activity": (
        "Catholic parishes have organized educational initiatives and "
        "community activities to encourage religious participation "
        "and preserve their traditions."
    ),

    "Unrelated control: technology": (
        "Modern semiconductor processors use transistors, logic gates "
        "and memory circuits to execute computational instructions."
    ),

    "Unrelated control: sports": (
        "The football team improved its passing accuracy and defensive "
        "coordination during the tournament."
    )
}


# ============================================================
# 3. LOAD TOKENIZER
# ============================================================

print("\n[1/5] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    use_fast=True
)

print("Tokenizer loaded.")


# ============================================================
# 4. LOAD MODEL
# ============================================================

print("\n[2/5] Checking existing Qwen2.5-3B model...")

if "model" not in globals():
    raise RuntimeError(
        "Qwen model is not loaded. Run the model-loading cell first."
    )

model.eval()

assert model.config.hidden_size == INPUT_DIM
assert model.config.num_hidden_layers == 36

print("Existing model reused successfully.")
print("Hidden dimension:", model.config.hidden_size)
print("Transformer layers:", model.config.num_hidden_layers)
print("GPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


# ============================================================
# 5. LOAD EXISTING SAE CHECKPOINT
# ============================================================

print("\n[3/5] Loading existing Layer 33 SAE...")

assert os.path.isfile(SAE_PATH), f"Missing checkpoint: {SAE_PATH}"

checkpoint = torch.load(
    SAE_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint["state_dict"]

W_enc = state["encoder.weight"]
b_enc = state["encoder.bias"]
mean = state["activation_mean"]

print("Checkpoint layer:", checkpoint.get("layer"))
print("Checkpoint config:", checkpoint.get("config"))

# PyTorch Linear weights: [latent_dim, input_dim]
assert tuple(W_enc.shape) == (LATENT_DIM, INPUT_DIM)
assert b_enc.numel() == LATENT_DIM
assert mean.numel() == INPUT_DIM

W_enc = W_enc.T.contiguous().to(
    DEVICE, dtype=torch.float32
)

b_enc = b_enc.reshape(-1).to(
    DEVICE, dtype=torch.float32
)

mean = mean.reshape(-1).to(
    DEVICE, dtype=torch.float32
)

print("SAE loaded successfully.")
print("Encoder:", tuple(W_enc.shape))
print("Bias:", tuple(b_enc.shape))
print("Mean:", tuple(mean.shape))
print("Target feature:", FEATURE)

del checkpoint, state
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 6. TARGETED FEATURE ACTIVATION EXPERIMENT
# ============================================================

print("\n[4/5] Running targeted activation experiment...")

results = []

with torch.inference_mode():

    for index, (category, sentence) in enumerate(examples.items()):

        print(f"\nProcessing [{index + 1}/{len(examples)}]: {category}")

        inputs = tokenizer(
            sentence,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        # ----------------------------------------------------
        # MODEL FORWARD PASS
        # ----------------------------------------------------

        output = model(
            **inputs,
            output_hidden_states=True,
            use_cache=False
        )

        # hidden_states[0] = embedding output
        # hidden_states[34] = output of transformer block 33

        residual = output.hidden_states[LAYER + 1].float()

        # ----------------------------------------------------
        # SAE ENCODER
        # ----------------------------------------------------

        pre_activations = (
            (residual - mean) @ W_enc + b_enc
        )

        # Raw ReLU activations.
        raw_activations = F.relu(pre_activations)

        # ----------------------------------------------------
        # TOP-K SPARSIFICATION
        # ----------------------------------------------------

        top_values, top_indices = torch.topk(
            raw_activations,
            k=TOP_K,
            dim=-1
        )

        sparse_activations = torch.zeros_like(raw_activations)

        sparse_activations.scatter_(
            -1,
            top_indices,
            top_values
        )

        # ----------------------------------------------------
        # EXTRACT TARGET FEATURE
        # ----------------------------------------------------

        valid_length = int(
            inputs["attention_mask"][0].sum().item()
        )

        raw_feature_values = raw_activations[
            0, :valid_length, FEATURE
        ]

        topk_feature_values = sparse_activations[
            0, :valid_length, FEATURE
        ]

        # Raw feature statistics.
        raw_max = raw_feature_values.max().item()
        raw_mean = raw_feature_values.mean().item()

        raw_active = int(
            (raw_feature_values > 0).sum().item()
        )

        # Top-K feature statistics.
        topk_max = topk_feature_values.max().item()
        topk_mean = topk_feature_values.mean().item()

        topk_active = int(
            (topk_feature_values > 0).sum().item()
        )

        # Strongest activation position.
        top_position = raw_feature_values.argmax().item()

        token_ids = inputs["input_ids"][0]

        strongest_token = tokenizer.decode(
            [token_ids[top_position].item()]
        )

        # Save complete token-level evidence.
        token_records = []

        for position in range(valid_length):

            token_records.append({
                "position": position,
                "token": tokenizer.decode(
                    [token_ids[position].item()]
                ),
                "raw_activation": float(
                    raw_feature_values[position].item()
                ),
                "topk_activation": float(
                    topk_feature_values[position].item()
                )
            })

        results.append({

            "context": category,
            "sentence": sentence,

            "raw_max_activation": raw_max,
            "raw_mean_activation": raw_mean,
            "raw_active_tokens": raw_active,

            "topk_max_activation": topk_max,
            "topk_mean_activation": topk_mean,
            "topk_active_tokens": topk_active,

            "total_tokens": valid_length,

            "strongest_token": strongest_token,
            "strongest_token_position": top_position,

            "token_records": token_records
        })

        print("Raw maximum:", f"{raw_max:.6f}")
        print("Raw active tokens:", raw_active, "/", valid_length)

        print("Top-K maximum:", f"{topk_max:.6f}")
        print("Top-K active tokens:", topk_active, "/", valid_length)

        print("Strongest token:", repr(strongest_token))

        # ----------------------------------------------------
        # MEMORY CLEANUP
        # ----------------------------------------------------

        del output
        del residual
        del pre_activations
        del raw_activations
        del sparse_activations
        del top_values
        del top_indices
        del inputs

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 95)
print("L33:F374 — TARGETED POLYSEMANTICITY INVESTIGATION")
print("=" * 95)

for result in results:

    print(f"\nContext: {result['context']}")

    print(
        "Raw maximum activation:",
        f"{result['raw_max_activation']:.6f}"
    )

    print(
        "Raw mean activation:",
        f"{result['raw_mean_activation']:.6f}"
    )

    print(
        "Raw active tokens:",
        f"{result['raw_active_tokens']} / {result['total_tokens']}"
    )

    print(
        "Top-K maximum activation:",
        f"{result['topk_max_activation']:.6f}"
    )

    print(
        "Top-K active tokens:",
        f"{result['topk_active_tokens']} / {result['total_tokens']}"
    )

    print(
        "Strongest token:",
        repr(result["strongest_token"])
    )

    print("-" * 95)


# ============================================================
# 8. EXPERIMENT SUMMARY
# ============================================================

raw_positive_contexts = sum(
    result["raw_active_tokens"] > 0
    for result in results
)

topk_positive_contexts = sum(
    result["topk_active_tokens"] > 0
    for result in results
)

print("\nEXPERIMENT SUMMARY")
print("=" * 95)

print("Target feature:", f"L{LAYER}:F{FEATURE}")
print("Tested contexts:", len(results))

print(
    "Contexts with raw activation:",
    raw_positive_contexts
)

print(
    "Contexts with Top-K activation:",
    topk_positive_contexts
)

print(
    "Contexts without raw activation:",
    len(results) - raw_positive_contexts
)

print(
    "Contexts without Top-K activation:",
    len(results) - topk_positive_contexts
)


# ============================================================
# 9. SAVE RESULTS
# ============================================================

print("\n[5/5] Saving results...")

# Full token-level JSON evidence.
json_path = os.path.join(
    OUTPUT_DIR,
    "L33_F374_targeted_activation_results.json"
)

with open(json_path, "w", encoding="utf-8") as file:

    json.dump(
        {
            "model": MODEL_ID,
            "revision": REVISION,
            "layer": LAYER,
            "feature": FEATURE,
            "top_k": TOP_K,
            "max_length": MAX_LENGTH,
            "results": results
        },
        file,
        indent=2,
        ensure_ascii=False
    )


# Compact CSV summary.
import pandas as pd

summary_rows = []

for result in results:

    summary_rows.append({

        "context": result["context"],

        "raw_max_activation": result["raw_max_activation"],
        "raw_mean_activation": result["raw_mean_activation"],
        "raw_active_tokens": result["raw_active_tokens"],

        "topk_max_activation": result["topk_max_activation"],
        "topk_mean_activation": result["topk_mean_activation"],
        "topk_active_tokens": result["topk_active_tokens"],

        "total_tokens": result["total_tokens"],

        "strongest_token": result["strongest_token"],
        "strongest_token_position": result["strongest_token_position"]
    })


df = pd.DataFrame(summary_rows)

csv_path = os.path.join(
    OUTPUT_DIR,
    "L33_F374_targeted_activation_summary.csv"
)

df.to_csv(csv_path, index=False)

print("JSON saved:", json_path)
print("CSV saved:", csv_path)

print("\nExperiment completed successfully.")
